# Spatia 100-video stable notebook — v3

Patch v3 fixes a Stage 2 failure mode where a validation exception could leave `model.stage='eval'`, disabling LoRA gradients on the next training batch and causing `loss.backward()` to fail with `element 0 of tensors does not require grad`.


# Spatia Wan2.2 Training Pipeline — 100 Videos / RTX 6000 Pro Stable

Notebook này được chỉnh cho yêu cầu train trên **100 video** với GPU VRAM lớn, ưu tiên chạy hết pipeline không NaN:
- LoRA rank 64, alpha 128.
- BF16 nếu GPU hỗ trợ, fallback FP16 + GradScaler.
- Stage 2 LR thấp, warmup/cosine, gradient clipping, timestep clamp, latent/prediction guard.
- Nếu không có 20 video validation riêng, notebook dùng 5 mẫu train làm validation proxy để không fail khi chỉ có đúng 100 video.

> Lưu ý: đây là bản stable reproduction cho 100 video, không phải full-scale paper 100k+ clips / 720p.


**Patch v2:** Fix validation/reconstruction dtype mismatch by explicitly casting Wan transformer inputs to BF16/backbone dtype.


## Stage 0.1 — Install / Check Dependencies


In [1]:
import os, sys, subprocess, importlib.util

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

REQUIRED_PACKAGES = [
    "tqdm",
    "opencv-python",
    "pandas",
    "numpy",
    "Pillow",
    "imageio",
    "safetensors",
]

# Không auto-install optional packages vì Kaggle có thể treo/lâu.
# Nếu thiếu, các stage Keye/ReferDINO/MapAnything sẽ fallback hoặc báo warning.
OPTIONAL_PACKAGES = [
    "transformers",
    "accelerate",
    "keye-vl-utils",
    "ruamel.yaml",
    "easydict",
]

def module_name(pkg):
    mapping = {
        "opencv-python": "cv2",
        "Pillow": "PIL",
        "ruamel.yaml": "ruamel",
        "keye-vl-utils": "keye_vl_utils",
    }
    return mapping.get(pkg, pkg.replace("-", "_"))

def pip_install(pkg):
    print("Installing", pkg)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--no-input",
        "-q",
        pkg,
    ])

print("Checking required packages...")
for pkg in REQUIRED_PACKAGES:
    mod = module_name(pkg)
    if importlib.util.find_spec(mod) is None:
        try:
            pip_install(pkg)
        except Exception as e:
            print("WARN required install failed:", pkg, type(e).__name__, e)
    else:
        print("OK", pkg)

print("\nChecking optional packages; no auto-install.")
for pkg in OPTIONAL_PACKAGES:
    mod = module_name(pkg)
    ok = importlib.util.find_spec(mod) is not None
    print(("OK       " if ok else "MISSING  "), pkg)

print("\nDone. Missing optional packages are allowed.")

Checking required packages...
OK tqdm
OK opencv-python
OK pandas
OK numpy
OK Pillow
OK imageio
OK safetensors

Checking optional packages; no auto-install.
OK        transformers
OK        accelerate
MISSING   keye-vl-utils
OK        ruamel.yaml
OK        easydict

Done. Missing optional packages are allowed.


## Stage 0.2 — Global Config


In [2]:
from pathlib import Path
import os, json, math, random, shutil, csv, gc, re, time, subprocess, inspect
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

# ============================================================
# 100-VIDEO / RTX 6000 PRO 95GB STABLE CONFIG
# ============================================================

# Requirement from assignment: train on 100 videos only.
TRAIN_VIDEOS = 100

# Optional validation. If the dataset only contains exactly 100 processed samples,
# the notebook automatically uses a small train-proxy validation subset instead of failing.
TEST_VIDEOS = 20
MAX_VIDEOS = TRAIN_VIDEOS + TEST_VIDEOS

# Stable single-GPU reproduction resolution.
# Paper uses much higher resolution, but 192x320 is kept to make the whole notebook finish reliably.
HEIGHT = 192
WIDTH = 320

# Paper-conditioned previous frames.
PREV_FRAMES = 9

# Wan-style video VAEs commonly prefer frame count 4n+1. 49 = 4*12+1.
# This keeps training stable on 100 clips while preserving a non-trivial video horizon.
TARGET_FRAMES = 49

# Candidate/reference setting.
CANDIDATE_FRAMES = 16
REF_FRAMES = 7
TOTAL_SAMPLE_FRAMES = CANDIDATE_FRAMES + PREV_FRAMES + TARGET_FRAMES

DEFAULT_PROMPT = "A realistic real estate video with smooth camera movement."
DEFAULT_KEYE_PROMPT = DEFAULT_PROMPT
USE_PROMPT_CSV_IF_FOUND = True

# External modules. Strict mode prevents silent fallback in preprocessing.
RUN_KEYE = False
RUN_REFERDINO = True
RUN_MAPANYTHING = True
STRICT_EXTERNAL_MODELS = True

# Backbone strictness. If Wan2.2 cannot load, Stage 4 must fail.
USE_WAN2_BACKBONE = True
ALLOW_TOY_ADAPTER = False
STRICT_WAN_BACKBONE = True

# Mixed precision.
# Prefer BF16 on RTX 6000 Pro / Blackwell/Ada/Hopper-class cards to reduce Stage-2 NaN risk.
def _pick_amp_dtype():
    if DEVICE.type != "cuda":
        return torch.float32
    try:
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16
    except Exception:
        pass
    return torch.float16

AMP_DTYPE = _pick_amp_dtype()
BACKBONE_DTYPE = AMP_DTYPE if DEVICE.type == "cuda" else torch.float32
USE_GRAD_SCALER = bool(DEVICE.type == "cuda" and AMP_DTYPE == torch.float16)

# Directories
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/workspace/outputs/spatia_full_work")
RUN_TAG = f"wan2_p{PREV_FRAMES}_t{TARGET_FRAMES}_c{CANDIDATE_FRAMES}_r{REF_FRAMES}_100vid_stable"

CACHE_DIR = WORK_DIR / f"spatia_full_cache_{RUN_TAG}"
PROC_DIR = WORK_DIR / f"processed_spatia_full_{RUN_TAG}"
CKPT_DIR = WORK_DIR / f"spatia_full_checkpoints_{RUN_TAG}"
SAMPLE_DIR = WORK_DIR / f"spatia_full_samples_{RUN_TAG}"
for d in [CACHE_DIR, PROC_DIR, CKPT_DIR, SAMPLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Training knobs
BATCH_SIZE = 1
NUM_WORKERS = 2 if os.name != "nt" else 0

# Effective batch 4 keeps the run stable without increasing per-forward VRAM.
GRAD_ACCUM_STEPS = 4

# Paper LR is too aggressive for 100-video LoRA fine-tuning. These values prioritize no-NaN completion.
LR_STAGE1 = 5e-6
LR_STAGE2 = 1e-6

# Paper reference values are kept for logging/audit only.
PAPER_STAGE1_STEPS = 8000
PAPER_STAGE2_STEPS = 5000

# Practical 100-video run. Increase only after one clean full run.
MAX_TRAIN_STEPS_STAGE1 = 800
MAX_TRAIN_STEPS_STAGE2 = 500

LOG_EVERY = 25
VAL_EVERY = 100
SAVE_EVERY = 100

# Checkpoint storage guard. Keep only the newest 1-2 checkpoints to avoid filling /kaggle/working.
KEEP_LAST_CKPTS = 1
MIN_FREE_GB_FOR_SAVE = 1.0

EVAL_MAX_BATCHES = 1

# Wan LoRA rank aligned with the paper audit.
LORA_RANK = 64
LORA_ALPHA = 128
LORA_DROPOUT = 0.05

# Stable latent control branch.
CONTROL_HIDDEN_MULT = 2  # legacy; not used by the stronger control branch
CONTROL_WIDTH = 384
CONTROL_DEPTH = 6
CONTROL_OUTPUT_SCALE = 0.50

# Optimizer / numerical guards
GRAD_CLIP_STAGE1 = 0.50
GRAD_CLIP_STAGE2 = 0.25
OPTIM_WEIGHT_DECAY = 1e-4
OPTIM_EPS = 1e-6
WARMUP_RATIO = 0.10
MIN_LR_SCALE = 0.10
MAX_CONSECUTIVE_BAD_STEPS = 25

# Flow-matching stability guards
TIMESTEP_MIN = 0.02
TIMESTEP_MAX = 0.98
LATENT_CLAMP_VALUE = 8.0
NOISE_CLAMP_VALUE = 4.0
PRED_CLAMP_VALUE = 10.0
LOSS_DIFF_CLAMP_VALUE = 5.0

CONTROL_LOSS_STATIC_WEIGHT = 1.0
CONTROL_LOSS_DYNAMIC_WEIGHT = 0.75

ENABLE_GRADIENT_CHECKPOINTING = True

print("Work dir:", WORK_DIR)
print("Run tag:", RUN_TAG)
print("Cache dir:", CACHE_DIR)
print("Proc dir:", PROC_DIR)
print("TOTAL_SAMPLE_FRAMES:", TOTAL_SAMPLE_FRAMES)
print("Train videos:", TRAIN_VIDEOS, "Optional val videos:", TEST_VIDEOS)
print("Effective batch:", BATCH_SIZE * GRAD_ACCUM_STEPS)
print("AMP dtype:", AMP_DTYPE, "GradScaler:", USE_GRAD_SCALER)
print("Wan backbone required:", USE_WAN2_BACKBONE, "strict:", STRICT_WAN_BACKBONE)


Device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Work dir: /kaggle/working
Run tag: wan2_p9_t49_c16_r7_100vid_stable
Cache dir: /kaggle/working/spatia_full_cache_wan2_p9_t49_c16_r7_100vid_stable
Proc dir: /kaggle/working/processed_spatia_full_wan2_p9_t49_c16_r7_100vid_stable
TOTAL_SAMPLE_FRAMES: 74
Train videos: 100 Optional val videos: 20
Effective batch: 4
AMP dtype: torch.bfloat16 GradScaler: False
Wan backbone required: True strict: True


## Patch ghi checkpoint

Bản này đã sửa phần lưu checkpoint: đưa tensor về CPU trước khi ghi file, ghi qua file `.tmp` rồi đổi tên, xóa checkpoint cũ và chỉ giữ `KEEP_LAST_CKPTS = 2` file mới nhất cho mỗi stage.


In [3]:
from pathlib import Path
import sys, subprocess, os, shutil, zipfile, importlib.util, importlib

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/workspace")

print("Top-level /kaggle/input:")
if KAGGLE_INPUT.exists():
    for p in sorted(KAGGLE_INPUT.iterdir()):
        print(" -", p)
else:
    print(" - /kaggle/input not found in this environment")

def is_referdino_repo(p):
    if p is None:
        return False
    p = Path(p)
    return (
        (p / "models" / "GroundingDINO" / "ops" / "setup.py").exists()
        and (p / "models").exists()
    )

def is_patched_referdino_repo(p):
    if p is None:
        return False
    return is_referdino_repo(p) and (Path(p) / "kaggle_bootstrap_referdino.py").exists()

def find_dir_contains(required_names, name_hint=None, root=KAGGLE_INPUT):
    hits = []
    if not root.exists():
        return hits

    for p in root.rglob("*"):
        if not p.is_dir():
            continue

        if name_hint and name_hint.lower() not in str(p).lower():
            continue

        ok = True
        for name in required_names:
            if not (p / name).exists():
                ok = False
                break

        if ok:
            hits.append(p)

    hits = sorted(
        hits,
        key=lambda x: (
            0 if (x / "kaggle_bootstrap_referdino.py").exists() else 1,
            len(x.parts),
            str(x)
        )
    )
    return hits

def find_patched_referdino_repo():
    """Prefer the new Kaggle-ready ReferDINO repo. Also supports a zip uploaded as a dataset."""
    if not KAGGLE_INPUT.exists():
        return None

    # 1. Best case: Kaggle extracted the dataset and the bootstrap file is visible.
    direct_hits = []
    for f in KAGGLE_INPUT.rglob("kaggle_bootstrap_referdino.py"):
        repo = f.parent
        if is_patched_referdino_repo(repo):
            direct_hits.append(repo)

    if direct_hits:
        return sorted(direct_hits, key=lambda p: (len(p.parts), str(p)))[0]

    # 2. If dataset contains the zip itself, extract it to /kaggle/working.
    zip_hits = [
        z for z in KAGGLE_INPUT.rglob("*.zip")
        if any(k in str(z).lower() for k in ["refer", "dino"])
    ]

    for z in sorted(zip_hits, key=lambda p: (len(p.parts), str(p))):
        extract_root = KAGGLE_WORKING / "extracted_inputs" / z.stem
        marker = extract_root / ".extracted_ok"

        try:
            if not marker.exists():
                if extract_root.exists():
                    shutil.rmtree(extract_root)

                extract_root.mkdir(parents=True, exist_ok=True)

                with zipfile.ZipFile(z, "r") as zipf:
                    zipf.extractall(extract_root)

                marker.write_text("ok", encoding="utf-8")

            for f in extract_root.rglob("kaggle_bootstrap_referdino.py"):
                repo = f.parent
                if is_patched_referdino_repo(repo):
                    print("Extracted patched ReferDINO zip:", z)
                    return repo

        except Exception as e:
            print("WARN cannot extract ReferDINO zip:", z, type(e).__name__, e)

    # 3. Fallback: old/unpatched ReferDINO repo layout.
    generic_hits = [
        p for p in find_dir_contains(["models", "configs"], "refer")
        if is_referdino_repo(p)
    ]

    if not generic_hits:
        generic_hits = [
            p for p in find_dir_contains(["models"], "refer")
            if is_referdino_repo(p)
        ]

    return generic_hits[0] if generic_hits else None

# ReferDINO: input repo is read-only; next cell will copy/build it into /kaggle/working.
REFERDINO_INPUT_REPO = find_patched_referdino_repo()
REFERDINO_REPO = REFERDINO_INPUT_REPO

# MapAnything repo root should contain mapanything/ package.
map_hits = (
    find_dir_contains(["mapanything", "pyproject.toml"], "map")
    or find_dir_contains(["mapanything", "setup.py"], "map")
    or find_dir_contains(["mapanything"], "map")
)

MAPANYTHING_REPO = map_hits[0] if map_hits else None

REFERDINO_CKPT = (
    next(KAGGLE_INPUT.rglob("ryt_mevis_swinb.pth"), None)
    if KAGGLE_INPUT.exists()
    else None
)

map_model_candidates = []
if KAGGLE_INPUT.exists():
    map_model_candidates = (
        list(KAGGLE_INPUT.rglob("Map-anything-v1"))
        + list(KAGGLE_INPUT.rglob("map-anything-v1"))
        + list(KAGGLE_INPUT.rglob("map_anything_v1"))
    )

MAPANYTHING_MODEL = map_model_candidates[0] if map_model_candidates else None

print("\nReferDINO input repo:", REFERDINO_INPUT_REPO)
print("ReferDINO patched:", is_patched_referdino_repo(REFERDINO_INPUT_REPO))
print("MapAnything repo:", MAPANYTHING_REPO)
print("ReferDINO ckpt:", REFERDINO_CKPT)
print("MapAnything model:", MAPANYTHING_MODEL)

# Important:
# Do NOT run `pip install -e MAPANYTHING_REPO` here.
# Kaggle input is read-only and editable install can hang.
# Use sys.path import only. If import fails, notebook will fallback later.

if MAPANYTHING_REPO is not None:
    map_repo = Path(MAPANYTHING_REPO)

    for p in [map_repo, map_repo.parent]:
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
            print("Added MapAnything path:", p)

try:
    importlib.invalidate_caches()
    import mapanything
    print("OK MapAnything import:", mapanything.__file__)
except Exception as e:
    print("WARN MapAnything import failed; will use fallback later:", type(e).__name__, e)

Top-level /kaggle/input:
 - /kaggle/input/competitions
 - /kaggle/input/datasets
 - /kaggle/input/models

ReferDINO input repo: /kaggle/input/datasets/nhtdngtrn/referdino-ready-on-kaggle-v2/ReferDINO-main
ReferDINO patched: True
MapAnything repo: /kaggle/input/datasets/nhtdngtrn/map-anything/map-anything-main
ReferDINO ckpt: /kaggle/input/models/nhtdngtrn/referdino-ryt-mevis-swinb/pytorch/default/1/ryt_mevis_swinb.pth
MapAnything model: /kaggle/input/models/nguynhunhtrngc/map-anything-v1/pytorch/default/1/Map-anything-v1
Added MapAnything path: /kaggle/input/datasets/nhtdngtrn/map-anything/map-anything-main
Added MapAnything path: /kaggle/input/datasets/nhtdngtrn/map-anything
OK MapAnything import: /kaggle/input/datasets/nhtdngtrn/map-anything/map-anything-main/mapanything/__init__.py


In [4]:

from pathlib import Path
import sys, os, subprocess, importlib.util

# Build/import patched ReferDINO for Kaggle.
# The patched repo contains kaggle_bootstrap_referdino.py, so the notebook no longer edits CUDA files manually.

def ensure_pkg(pkg, module=None):
    module = module or pkg
    if importlib.util.find_spec(module) is not None:
        return True
    try:
        print("Installing", pkg)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-input", pkg])
        return True
    except Exception as e:
        print("WARN install failed:", pkg, type(e).__name__, e)
        return False

# Small build helpers. These are usually already present on Kaggle, but this keeps the build cell self-contained.
ensure_pkg("ninja", "ninja")
ensure_pkg("cython", "Cython")
ensure_pkg("PyYAML", "yaml")

REFERDINO_BUILD_OK = False
REFERDINO_BUILD_ERROR = None
REFERDINO_OPS_DIR = None

if not RUN_REFERDINO:
    print("RUN_REFERDINO=False, skipping ReferDINO CUDA ops build.")
elif REFERDINO_INPUT_REPO is None:
    REFERDINO_BUILD_ERROR = "Patched ReferDINO repo not found in /kaggle/input"
    print("WARN", REFERDINO_BUILD_ERROR)
    if STRICT_EXTERNAL_MODELS:
        raise FileNotFoundError(REFERDINO_BUILD_ERROR)
else:
    src_repo = Path(REFERDINO_INPUT_REPO)
    print("Bootstrapping ReferDINO from:", src_repo)
    if str(src_repo) not in sys.path:
        sys.path.insert(0, str(src_repo))
    try:
        from kaggle_bootstrap_referdino import bootstrap
        # bootstrap copies read-only Kaggle input to /kaggle/working, sets sys.path, builds ops, and verifies imports.
        REFERDINO_REPO = bootstrap(src_repo, jobs=2, verbose=True)
        REFERDINO_OPS_DIR = Path(REFERDINO_REPO) / "models" / "GroundingDINO" / "ops"
        for p in [REFERDINO_REPO, Path(REFERDINO_REPO) / "models" / "GroundingDINO", REFERDINO_OPS_DIR]:
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
        REFERDINO_BUILD_OK = True
        print("OK ReferDINO CUDA ops built and imported.")
    except Exception as e:
        REFERDINO_BUILD_ERROR = f"{type(e).__name__}: {e}"
        print("WARN ReferDINO bootstrap/build failed:", REFERDINO_BUILD_ERROR)
        print("The pipeline can still continue with optical-flow masks unless STRICT_EXTERNAL_MODELS=True.")
        REFERDINO_REPO = src_repo
        if STRICT_EXTERNAL_MODELS:
            raise

# Lightweight final import check.
try:
    if REFERDINO_REPO is not None and str(REFERDINO_REPO) not in sys.path:
        sys.path.insert(0, str(REFERDINO_REPO))
    import MultiScaleDeformableAttention
    print("OK MultiScaleDeformableAttention import")
except Exception as e:
    print("WARN MultiScaleDeformableAttention import failed:", type(e).__name__, e)

try:
    from models import build_model
    print("OK ReferDINO models import")
except Exception as e:
    print("WARN ReferDINO models import failed:", type(e).__name__, e)


Bootstrapping ReferDINO from: /kaggle/input/datasets/nhtdngtrn/referdino-ready-on-kaggle-v2/ReferDINO-main
enced
      const int q_col = _temp % num_query;
                ^
          detected during instantiation of "void ms_deformable_col2im_cuda(cudaStream_t, const scalar_t *, const scalar_t *, const int64_t *, const int64_t *, const scalar_t *, const scalar_t *, int, int, int, int, int, int, int, scalar_t *, scalar_t *, scalar_t *) [with scalar_t=double]" at line 134 of /kaggle/working/external_repos/ReferDINO-main/models/GroundingDINO/ops/src/cuda/ms_deform_attn_cuda.cu

/kaggle/working/external_repos/ReferDINO-main/models/GroundingDINO/ops/src/cuda/ms_deform_im2col_cuda.cuh(872): warning #177-D: variable "q_col" was declared but never referenced
      const int q_col = _temp % num_query;
                ^
          detected during instantiation of "void ms_deformable_col2im_cuda(cudaStream_t, const scalar_t *, const scalar_t *, const int64_t *, const int64_t *, const scalar_t *, 

/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/kaggle/working/external_repos/ReferDINO-main/models/GroundingDINO/utils.py:66: SyntaxWarning: invalid escape sequence '\s'
  - memory: bs, \sum{hw}, d_model


ModuleNotFoundError: No module named 'loralib'

In [5]:
from pathlib import Path
import sys, textwrap, os

OFFLINE_PKGS = Path("/kaggle/working/offline_pkgs")
OFFLINE_PKGS.mkdir(parents=True, exist_ok=True)

if str(OFFLINE_PKGS) not in sys.path:
    sys.path.insert(0, str(OFFLINE_PKGS))

# 1. Minimal loralib fallback for ReferDINO
(OFFLINE_PKGS / "loralib.py").write_text(r'''
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class Linear(nn.Linear):
    def __init__(
        self,
        in_features,
        out_features,
        r=0,
        lora_alpha=1,
        lora_dropout=0.0,
        fan_in_fan_out=False,
        merge_weights=True,
        **kwargs
    ):
        super().__init__(in_features, out_features, **kwargs)
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = lora_alpha / r if r and r > 0 else 1
        self.fan_in_fan_out = fan_in_fan_out
        self.merge_weights = merge_weights
        self.merged = False
        self.lora_dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0 else nn.Identity()

        if r and r > 0:
            self.lora_A = nn.Parameter(torch.zeros((r, in_features)))
            self.lora_B = nn.Parameter(torch.zeros((out_features, r)))
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B)

    def forward(self, x):
        result = F.linear(x, self.weight, self.bias)
        if getattr(self, "r", 0) and self.r > 0 and hasattr(self, "lora_A") and hasattr(self, "lora_B"):
            after_A = F.linear(self.lora_dropout(x), self.lora_A)
            result = result + F.linear(after_A, self.lora_B) * self.scaling
        return result

def mark_only_lora_as_trainable(model, bias="none"):
    for n, p in model.named_parameters():
        p.requires_grad = "lora_" in n

def lora_state_dict(model, bias="none"):
    return {k: v for k, v in model.state_dict().items() if "lora_" in k}
''', encoding="utf-8")

# 2. Minimal addict fallback
(OFFLINE_PKGS / "addict.py").write_text(r'''
class Dict(dict):
    def __init__(self, *args, **kwargs):
        super().__init__()
        data = dict(*args, **kwargs)
        for k, v in data.items():
            self[k] = self._wrap(v)

    @classmethod
    def _wrap(cls, v):
        if isinstance(v, dict) and not isinstance(v, Dict):
            return cls(v)
        if isinstance(v, list):
            return [cls._wrap(x) for x in v]
        return v

    def __getattr__(self, name):
        try:
            return self[name]
        except KeyError:
            raise AttributeError(name)

    def __setattr__(self, name, value):
        self[name] = self._wrap(value)
''', encoding="utf-8")

# 3. Minimal termcolor fallback
(OFFLINE_PKGS / "termcolor.py").write_text(r'''
def colored(text, color=None, on_color=None, attrs=None):
    return text
''', encoding="utf-8")

# 4. Minimal colorlog fallback
(OFFLINE_PKGS / "colorlog.py").write_text(r'''
import logging
class ColoredFormatter(logging.Formatter):
    pass
''', encoding="utf-8")

# 5. Minimal yapf fallback
yapf_dir = OFFLINE_PKGS / "yapf" / "yapflib"
yapf_dir.mkdir(parents=True, exist_ok=True)
(OFFLINE_PKGS / "yapf" / "__init__.py").write_text("", encoding="utf-8")
(yapf_dir / "__init__.py").write_text("", encoding="utf-8")
(yapf_dir / "yapf_api.py").write_text(r'''
def FormatCode(code, *args, **kwargs):
    return code, False
''', encoding="utf-8")

print("Offline fallback packages created at:", OFFLINE_PKGS)

# Point back to built ReferDINO repo
REFERDINO_REPO = Path("/kaggle/working/external_repos/ReferDINO-main")
REFERDINO_OPS_DIR = REFERDINO_REPO / "models" / "GroundingDINO" / "ops"

for p in [
    OFFLINE_PKGS,
    REFERDINO_REPO,
    REFERDINO_REPO / "models" / "GroundingDINO",
    REFERDINO_OPS_DIR,
]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        print("Added path:", p)

# Import check
try:
    import loralib
    print("OK loralib offline:", loralib.__file__)
except Exception as e:
    print("ERR loralib offline failed:", type(e).__name__, e)
    raise

try:
    import MultiScaleDeformableAttention
    print("OK MultiScaleDeformableAttention import")
except Exception as e:
    print("ERR MultiScaleDeformableAttention import failed:", type(e).__name__, e)
    raise

try:
    from models import build_model
    print("OK ReferDINO models import")
    REFERDINO_BUILD_OK = True
    REFERDINO_BUILD_ERROR = None
except Exception as e:
    REFERDINO_BUILD_OK = False
    REFERDINO_BUILD_ERROR = f"{type(e).__name__}: {e}"
    print("ERR ReferDINO models import failed:", REFERDINO_BUILD_ERROR)
    raise

print("Final REFERDINO_REPO:", REFERDINO_REPO)
print("REFERDINO_BUILD_OK:", REFERDINO_BUILD_OK)

Offline fallback packages created at: /kaggle/working/offline_pkgs
OK loralib offline: /kaggle/working/offline_pkgs/loralib.py
OK MultiScaleDeformableAttention import
OK ReferDINO models import
Final REFERDINO_REPO: /kaggle/working/external_repos/ReferDINO-main
REFERDINO_BUILD_OK: True


## Stage 1.1 — Auto-detect Kaggle Inputs


In [6]:

KAGGLE_INPUT_ROOT = Path("/kaggle/input")

def list_input_roots(input_root=KAGGLE_INPUT_ROOT):
    if not input_root.exists():
        return []
    return sorted([p for p in input_root.iterdir() if p.is_dir()])

def has_any_file(root, patterns):
    if root is None:
        return False
    root = Path(root)
    if root.is_file():
        return any(root.match(pattern) for pattern in patterns)
    if not patterns:
        return True
    for pattern in patterns:
        if any(root.rglob(pattern)):
            return True
    return False

def find_dir_by_keywords(keywords, must_have_any=None, input_root=KAGGLE_INPUT_ROOT, recursive=True):
    keywords = [k.lower() for k in keywords]
    search_dirs = list_input_roots(input_root)
    if recursive:
        expanded = []
        for root in search_dirs:
            expanded.append(root)
            expanded.extend([p for p in root.rglob("*") if p.is_dir()])
        search_dirs = expanded
    candidates = []
    for root in search_dirs:
        text = str(root).lower()
        if all(k in text for k in keywords) and has_any_file(root, must_have_any):
            candidates.append(root)
    candidates = sorted(candidates, key=lambda p: (len(p.parts), str(p)))
    return candidates[0] if candidates else None

def find_file_by_keywords(keywords, patterns, input_root=KAGGLE_INPUT_ROOT):
    keywords = [k.lower() for k in keywords]
    candidates = []
    if not input_root.exists():
        return None
    for pattern in patterns:
        for f in input_root.rglob(pattern):
            text = str(f).lower()
            if all(k in text for k in keywords):
                candidates.append(f)
    candidates = sorted(candidates, key=lambda p: (len(p.parts), str(p)))
    return candidates[0] if candidates else None

def prefer_existing_path(var_name, detected):
    old = globals().get(var_name)
    if old is not None and Path(old).exists():
        return Path(old)
    return detected

print("Kaggle input roots:")
for p in list_input_roots():
    print(" -", p)

DATA_ROOT = (
    find_dir_by_keywords(["realestate"], ["*.mp4", "*.txt", "*.csv"])
    or find_dir_by_keywords(["real", "estate"], ["*.mp4", "*.txt", "*.csv"])
    or find_dir_by_keywords(["small", "dataset"], ["*.mp4", "*.txt", "*.csv"])
)

# Do not overwrite the writable, already-bootstrapped ReferDINO path from cell 6.
_detected_refer_input = find_patched_referdino_repo() if "find_patched_referdino_repo" in globals() else find_dir_by_keywords(["referdino"], ["*.py"])
REFERDINO_INPUT_REPO = prefer_existing_path("REFERDINO_INPUT_REPO", _detected_refer_input)
if globals().get("REFERDINO_BUILD_OK", False) and globals().get("REFERDINO_REPO") is not None:
    REFERDINO_REPO = Path(globals()["REFERDINO_REPO"])
else:
    REFERDINO_REPO = prefer_existing_path("REFERDINO_REPO", REFERDINO_INPUT_REPO)

MAPANYTHING_REPO = prefer_existing_path("MAPANYTHING_REPO", find_dir_by_keywords(["map", "anything"], ["*.py"]))
WAN_MODEL = find_dir_by_keywords(["wan"], ["*.safetensors", "*.json", "*.pth"])
MAPANYTHING_MODEL = prefer_existing_path("MAPANYTHING_MODEL", (
    find_dir_by_keywords(["map", "anything", "v1"], ["*.safetensors", "*.bin", "*.pth"])
    or find_dir_by_keywords(["map", "anything"], ["*.safetensors", "*.bin", "*.pth"])
))
KEYE_MODEL = find_dir_by_keywords(["keye"], ["*.safetensors", "*.json", "*.py"])
REFERDINO_CKPT = prefer_existing_path("REFERDINO_CKPT", (
    find_file_by_keywords(["referdino"], ["*.pth", "*.pt", "*.ckpt"])
    or find_file_by_keywords(["mevis"], ["*.pth", "*.pt", "*.ckpt"])
    or find_file_by_keywords(["swin"], ["*.pth", "*.pt", "*.ckpt"])
))

print("\nDetected paths:")
for name, path in {
    "DATA_ROOT": DATA_ROOT,
    "MAPANYTHING_REPO": MAPANYTHING_REPO,
    "REFERDINO_INPUT_REPO": REFERDINO_INPUT_REPO,
    "REFERDINO_REPO": REFERDINO_REPO,
    "REFERDINO_BUILD_OK": REFERDINO_BUILD_OK,
    "WAN_MODEL": WAN_MODEL,
    "MAPANYTHING_MODEL": MAPANYTHING_MODEL,
    "KEYE_MODEL": KEYE_MODEL,
    "REFERDINO_CKPT": REFERDINO_CKPT,
}.items():
    print(f"{name:22s} -> {path}")

if DATA_ROOT is None:
    raise FileNotFoundError("Cannot find RealEstate dataset under /kaggle/input.")


Kaggle input roots:
 - /kaggle/input/competitions
 - /kaggle/input/datasets
 - /kaggle/input/models

Detected paths:
DATA_ROOT              -> /kaggle/input/datasets/nhtdngtrn/realestate10k-130
MAPANYTHING_REPO       -> /kaggle/input/datasets/nhtdngtrn/map-anything/map-anything-main
REFERDINO_INPUT_REPO   -> /kaggle/input/datasets/nhtdngtrn/referdino-ready-on-kaggle-v2/ReferDINO-main
REFERDINO_REPO         -> /kaggle/working/external_repos/ReferDINO-main
REFERDINO_BUILD_OK     -> True
WAN_MODEL              -> /kaggle/input/models/nhtdngtrn/wan2-2-5b-diffuser
MAPANYTHING_MODEL      -> /kaggle/input/models/nguynhunhtrngc/map-anything-v1/pytorch/default/1/Map-anything-v1
KEYE_MODEL             -> /kaggle/input/models/nhtdngtrn/keye-vl-1-5-8b
REFERDINO_CKPT         -> /kaggle/input/models/nhtdngtrn/referdino-ryt-mevis-swinb/pytorch/default/1/ryt_mevis_swinb.pth


In [7]:
from pathlib import Path
import os, json, re, shutil

MODEL_ROOT = Path("/kaggle/input/models")

def find_dir_with_files(base, required_files, name_hint=None):
    base = Path(base)
    candidates = []

    for d in [base] + [p for p in base.rglob("*") if p.is_dir()]:
        score = 0
        for f in required_files:
            if (d / f).exists():
                score += 1

        if score == len(required_files):
            if name_hint and name_hint.lower() in str(d).lower():
                score += 2
            candidates.append((score, d))

    candidates = sorted(candidates, key=lambda x: (x[0], len(str(x[1]))), reverse=True)

    print(f"\nCandidates for {name_hint or required_files}:")
    for s, d in candidates[:10]:
        print(s, "->", d)

    if not candidates:
        raise FileNotFoundError(f"Cannot find directory under {base} with files: {required_files}")

    return candidates[0][1]


# 1. Fix Keye path
KEYE_BASE = Path("/kaggle/input/models/nhtdngtrn/keye-vl-1-5-8b")

KEYE_REAL_DIR = find_dir_with_files(
    KEYE_BASE,
    [
        "config.json",
        "configuration_keye_vl_1_5.py",
        "modeling_keye_vl_1_5.py",
        "processing_keye_vl_1_5.py",
        "model.safetensors.index.json",
    ],
    name_hint="keye"
)

print("\nKEYE_REAL_DIR:", KEYE_REAL_DIR)


# 2. Patch Keye config if Transformers cannot recognize it
# Use symlink folder in /kaggle/working, only config.json is copied/modified.
KEYE_FIXED_DIR = Path("/kaggle/working/keye_vl_1_5_fixed")

if KEYE_FIXED_DIR.exists():
    shutil.rmtree(KEYE_FIXED_DIR)

KEYE_FIXED_DIR.mkdir(parents=True, exist_ok=True)

for p in KEYE_REAL_DIR.iterdir():
    if p.name == "config.json":
        continue
    os.symlink(p, KEYE_FIXED_DIR / p.name)

cfg = json.loads((KEYE_REAL_DIR / "config.json").read_text())

conf_py = (KEYE_REAL_DIR / "configuration_keye_vl_1_5.py").read_text()
model_py = (KEYE_REAL_DIR / "modeling_keye_vl_1_5.py").read_text()

config_classes = re.findall(r"^class\s+(\w+)\s*\(", conf_py, flags=re.M)
config_classes = [c for c in config_classes if "Config" in c]

model_classes = re.findall(r"^class\s+(\w+)\s*\(", model_py, flags=re.M)

print("Config classes:", config_classes)
print("Model classes:", model_classes[:30])

if not config_classes:
    raise RuntimeError("Cannot find Config class inside configuration_keye_vl_1_5.py")

config_cls = config_classes[0]

preferred_model_classes = [
    c for c in model_classes
    if "ForConditionalGeneration" in c
    or "ForCausalLM" in c
    or "ForImageTextToText" in c
    or "Keye" in c
]

if cfg.get("architectures"):
    model_cls = cfg["architectures"][0]
elif preferred_model_classes:
    model_cls = preferred_model_classes[0]
else:
    raise RuntimeError("Cannot infer Keye model class from modeling_keye_vl_1_5.py")

cfg.setdefault("model_type", "keye_vl_1_5")
cfg["architectures"] = [model_cls]

auto_map = cfg.get("auto_map", {})
auto_map.setdefault("AutoConfig", f"configuration_keye_vl_1_5.{config_cls}")
auto_map.setdefault("AutoModel", f"modeling_keye_vl_1_5.{model_cls}")
auto_map.setdefault("AutoModelForCausalLM", f"modeling_keye_vl_1_5.{model_cls}")
auto_map.setdefault("AutoModelForVision2Seq", f"modeling_keye_vl_1_5.{model_cls}")
auto_map.setdefault("AutoModelForImageTextToText", f"modeling_keye_vl_1_5.{model_cls}")
cfg["auto_map"] = auto_map

with open(KEYE_FIXED_DIR / "config.json", "w") as f:
    json.dump(cfg, f, indent=2)

KEYE_MODEL = KEYE_FIXED_DIR

print("\nSelected KEYE_MODEL:", KEYE_MODEL)
print("model_type:", cfg.get("model_type"))
print("architectures:", cfg.get("architectures"))
print("auto_map:", json.dumps(cfg.get("auto_map"), indent=2))


# 3. Fix Wan path
WAN_BASE = Path("/kaggle/input/models/nhtdngtrn/wan2-2-5b-diffuser")

WAN_MODEL = find_dir_with_files(
    WAN_BASE,
    ["model_index.json"],
    name_hint="wan"
)

print("\nSelected WAN_MODEL:", WAN_MODEL)
print("Wan model_index exists:", (WAN_MODEL / "model_index.json").exists())


Candidates for keye:
7 -> /kaggle/input/models/nhtdngtrn/keye-vl-1-5-8b/pytorch/default/1

KEYE_REAL_DIR: /kaggle/input/models/nhtdngtrn/keye-vl-1-5-8b/pytorch/default/1
Config classes: ['KeyeVL1_5VisionConfig', 'KeyeVL1_5Config']
Model classes: ['Qwen3VisionPatchEmbed', 'Qwen3VisionRotaryEmbedding', 'KeyeVL1_5VisionModelOutput', 'KeyeVL1_5VisionEmbeddings', 'KeyeVL1_5VisionAttention', 'KeyeVL1_5VisionMLP', 'KeyeVL1_5VisionEncoderLayer', 'KeyeVL1_5VisionPreTrainedModel', 'KeyeVL1_5VisionEncoder', 'KeyeVL1_5VisionTransformer', 'KeyeVL1_5VisionModelMultiheadAttentionPoolingHead', 'KeyeVL1_5VisionModel', 'Qwen3RMSNorm', 'KeyeVL1_5PatchMerger', 'Qwen3PreTrainedModel', 'KeyeVL1_5VisionRotaryEmbedding', 'KeyeVL1_5RotaryEmbedding', 'Qwen3MLP', 'KeyeVL1_5Attention', 'KeyeVL1_5FlashAttention2', 'KeyeVL1_5SdpaAttention', 'KeyeVL1_5DecoderLayer', 'Qwen3Model', 'KeyeVL1_5CausalLMOutputWithPast', 'KeyeVL1_5ForConditionalGeneration', 'Projector']

Selected KEYE_MODEL: /kaggle/working/keye_vl_1_5_fi

In [8]:
from pathlib import Path
import sys, subprocess

def find_hydra_wheel_dir(root="/kaggle/input"):
    root = Path(root)
    candidates = []
    for p in root.rglob("*"):
        if p.is_dir():
            files = [x.name for x in p.iterdir() if x.suffix in [".whl", ".gz", ".zip"]]
            if any("hydra_core-1.3.2" in n for n in files):
                candidates.append(p)
    candidates = sorted(candidates, key=lambda x: (len(x.parts), str(x)))
    return candidates[0] if candidates else None

HYDRA_WHEEL_DIR = find_hydra_wheel_dir()
print("HYDRA_WHEEL_DIR:", HYDRA_WHEEL_DIR)

if HYDRA_WHEEL_DIR is None:
    raise FileNotFoundError("Không tìm thấy hydra_core-1.3.2 trong /kaggle/input")

subprocess.call([
    sys.executable, "-m", "pip", "uninstall",
    "-y", "hydra-core"
])

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-index",
    "--no-deps",
    f"--find-links={HYDRA_WHEEL_DIR}",
    "antlr4-python3-runtime==4.9.3",
    "omegaconf==2.3.0",
    "hydra-core==1.3.2",
    "wadler-lindig",
    "jaxtyping",
    "trimesh",
])

import hydra, omegaconf
print("hydra:", hydra.__version__)
print("omegaconf:", omegaconf.__version__)

assert hydra.__version__ == "1.3.2"
assert omegaconf.__version__ == "2.3.0"

print("OK Hydra/MapAnything deps offline")

HYDRA_WHEEL_DIR: /kaggle/input/datasets/nhtdngtrn/hydra-132-mapanything-wheels


Looking in links: /kaggle/input/datasets/nhtdngtrn/hydra-132-mapanything-wheels
Processing /kaggle/input/datasets/nhtdngtrn/hydra-132-mapanything-wheels/hydra_core-1.3.2-py3-none-any.whl
Processing /kaggle/input/datasets/nhtdngtrn/hydra-132-mapanything-wheels/wadler_lindig-0.1.7-py3-none-any.whl
Processing /kaggle/input/datasets/nhtdngtrn/hydra-132-mapanything-wheels/jaxtyping-0.3.11-py3-none-any.whl
Processing /kaggle/input/datasets/nhtdngtrn/hydra-132-mapanything-wheels/trimesh-4.12.2-py3-none-any.whl
hydra: 1.3.2
omegaconf: 2.3.0
OK Hydra/MapAnything deps offline


## Stage 1.2 — Add Repo Paths and Verify Assets


In [10]:

repo_paths = []
for repo in [MAPANYTHING_REPO, REFERDINO_REPO]:
    if repo is not None:
        repo = Path(repo)
        repo_paths.append(repo)
        if repo.name.lower().startswith("referdino") or (repo / "models" / "GroundingDINO").exists():
            repo_paths.extend([
                repo / "models" / "GroundingDINO",
                repo / "models" / "GroundingDINO" / "ops",
            ])

for repo in repo_paths:
    if repo is not None and repo.exists() and str(repo) not in sys.path:
        sys.path.insert(0, str(repo))
        print("Added to sys.path:", repo)

def preview_files(root, patterns, max_items=8):
    if root is None:
        return []
    root = Path(root)
    if root.is_file():
        return [root]
    files = []
    for pat in patterns:
        files.extend(root.rglob(pat))
    return sorted(set(files))[:max_items]

print("Wan files:", preview_files(WAN_MODEL, ["*.safetensors", "*.json", "*.pth"], 10))
print("MapAnything model files:", preview_files(MAPANYTHING_MODEL, ["*.safetensors", "*.bin", "*.pth"], 10))
print("Keye files:", preview_files(KEYE_MODEL, ["*.safetensors", "*.json", "*.py"], 10))
print("ReferDINO ckpt:", REFERDINO_CKPT)
print("ReferDINO build ok:", globals().get("REFERDINO_BUILD_OK", False))
print("ReferDINO build error:", globals().get("REFERDINO_BUILD_ERROR", None))
print("MapAnything repo py:", preview_files(MAPANYTHING_REPO, ["*.py"], 10))
print("ReferDINO repo py:", preview_files(REFERDINO_REPO, ["*.py"], 10))


Wan files: [PosixPath('/kaggle/input/models/nhtdngtrn/wan2-2-5b-diffuser/pytorch/default/1/model_index.json'), PosixPath('/kaggle/input/models/nhtdngtrn/wan2-2-5b-diffuser/pytorch/default/1/scheduler/scheduler_config.json'), PosixPath('/kaggle/input/models/nhtdngtrn/wan2-2-5b-diffuser/pytorch/default/1/text_encoder/config.json'), PosixPath('/kaggle/input/models/nhtdngtrn/wan2-2-5b-diffuser/pytorch/default/1/text_encoder/model-00001-of-00003.safetensors'), PosixPath('/kaggle/input/models/nhtdngtrn/wan2-2-5b-diffuser/pytorch/default/1/text_encoder/model-00002-of-00003.safetensors'), PosixPath('/kaggle/input/models/nhtdngtrn/wan2-2-5b-diffuser/pytorch/default/1/text_encoder/model-00003-of-00003.safetensors'), PosixPath('/kaggle/input/models/nhtdngtrn/wan2-2-5b-diffuser/pytorch/default/1/text_encoder/model.safetensors.index.json'), PosixPath('/kaggle/input/models/nhtdngtrn/wan2-2-5b-diffuser/pytorch/default/1/tokenizer/special_tokens_map.json'), PosixPath('/kaggle/input/models/nhtdngtrn/wa

## Stage 1.3 — Wan2.2 Asset Audit

Cell này chỉ kiểm tra asset. Stage 4 mới load weight Wan2.2 để tránh giữ model lớn trên GPU trong lúc preprocess.


In [11]:
from pathlib import Path
import json, os


def resolve_diffusers_model_dir(root, name="model"):
    root = Path(root) if root is not None else None
    if root is None or not root.exists():
        raise FileNotFoundError(f"{name} root not found: {root}")
    if (root / "model_index.json").exists():
        return root
    hits = sorted(root.rglob("model_index.json"), key=lambda p: (len(p.parts), str(p)))
    if not hits:
        raise FileNotFoundError(f"No model_index.json found under {root}")
    return hits[0].parent

WAN_DIR = resolve_diffusers_model_dir(WAN_MODEL, "Wan2.2")
print("WAN_DIR:", WAN_DIR)

model_index = json.loads((WAN_DIR / "model_index.json").read_text(encoding="utf-8"))
print("model_index keys:", sorted(model_index.keys()))
print("_class_name:", model_index.get("_class_name"))

required_any = ["transformer", "vae", "text_encoder", "tokenizer", "scheduler"]
missing = [name for name in required_any if not (WAN_DIR / name).exists()]
if missing:
    print("WARN missing standard component folders:", missing)
    print("Available dirs:", [p.name for p in WAN_DIR.iterdir() if p.is_dir()])
    if STRICT_WAN_BACKBONE:
        raise FileNotFoundError(f"Wan2.2 Diffusers folder is incomplete. Missing: {missing}")
else:
    print("Wan2.2 Diffusers asset audit OK.")


WAN_DIR: /kaggle/input/models/nhtdngtrn/wan2-2-5b-diffuser/pytorch/default/1
model_index keys: ['_class_name', '_diffusers_version', 'boundary_ratio', 'expand_timesteps', 'scheduler', 'text_encoder', 'tokenizer', 'transformer', 'transformer_2', 'vae']
_class_name: WanPipeline
Wan2.2 Diffusers asset audit OK.


## Stage 2.1 — Load RealEstate10K Videos, Poses, Prompts


In [12]:
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

def find_child_dir(root, name):
    matches = [p for p in Path(root).rglob(name) if p.is_dir()]
    return matches[0] if matches else None

VIDEO_DIR = find_child_dir(DATA_ROOT, "videos") or DATA_ROOT
POSE_DIR = find_child_dir(DATA_ROOT, "poses") or DATA_ROOT
manifest_candidates = list(Path(DATA_ROOT).rglob("manifest.csv"))
prompt_candidates = list(Path(DATA_ROOT).rglob("prompts.csv"))
MANIFEST_PATH = manifest_candidates[0] if manifest_candidates else None
PROMPT_CSV = prompt_candidates[0] if prompt_candidates else None

print("VIDEO_DIR:", VIDEO_DIR)
print("POSE_DIR:", POSE_DIR)
print("MANIFEST_PATH:", MANIFEST_PATH)
print("PROMPT_CSV:", PROMPT_CSV)

video_files = sorted([p for p in Path(VIDEO_DIR).rglob("*") if p.suffix.lower() in VIDEO_EXTS])
pose_files = sorted([p for p in Path(POSE_DIR).rglob("*.txt")])
pose_by_stem = {p.stem: p for p in pose_files}

prompt_by_stem = {}
if USE_PROMPT_CSV_IF_FOUND and PROMPT_CSV is not None:
    pdf = pd.read_csv(PROMPT_CSV)
    for _, row in pdf.iterrows():
        stem = Path(str(row.get("video_path", row.get("id", "")))).stem
        prompt = str(row.get("prompt", DEFAULT_PROMPT))
        if stem:
            prompt_by_stem[stem] = prompt

records = []
if MANIFEST_PATH is not None:
    mdf = pd.read_csv(MANIFEST_PATH)
    for _, row in mdf.iterrows():
        status = str(row.get("status", "ok"))
        if status not in ["ok", "exists", ""]:
            continue
        vp_raw = str(row.get("video_path", ""))
        pp_raw = str(row.get("pose_path", ""))
        stem = str(row.get("id", Path(vp_raw).stem))
        vp = Path(vp_raw)
        pp = Path(pp_raw)
        if not vp.exists():
            cand = [p for p in video_files if p.stem == stem]
            vp = cand[0] if cand else None
        if not pp.exists():
            pp = pose_by_stem.get(stem)
        if vp is not None and pp is not None and vp.exists() and pp.exists():
            records.append({"id": stem, "video_path": str(vp), "pose_path": str(pp), "prompt": prompt_by_stem.get(stem, DEFAULT_PROMPT)})
else:
    for vp in video_files:
        pp = pose_by_stem.get(vp.stem)
        if pp is not None:
            records.append({"id": vp.stem, "video_path": str(vp), "pose_path": str(pp), "prompt": prompt_by_stem.get(vp.stem, DEFAULT_PROMPT)})

records = records[:MAX_VIDEOS]
print("Usable video-pose pairs:", len(records))
for r in records[:5]:
    print(r)

if len(records) < min(10, MAX_VIDEOS):
    raise RuntimeError("Too few usable video/pose pairs. Check videos/poses naming and manifest.csv.")

VIDEO_DIR: /kaggle/input/datasets/nhtdngtrn/realestate10k-130/videos
POSE_DIR: /kaggle/input/datasets/nhtdngtrn/realestate10k-130/poses
MANIFEST_PATH: /kaggle/input/datasets/nhtdngtrn/realestate10k-130/manifest.csv
PROMPT_CSV: None
Usable video-pose pairs: 120
{'id': '00000_000c3ab189999a83', 'video_path': '/kaggle/input/datasets/nhtdngtrn/realestate10k-130/videos/00000_000c3ab189999a83.mp4', 'pose_path': '/kaggle/input/datasets/nhtdngtrn/realestate10k-130/poses/00000_000c3ab189999a83.txt', 'prompt': 'A realistic real estate video with smooth camera movement.'}
{'id': '00001_000db54a47bd43fe', 'video_path': '/kaggle/input/datasets/nhtdngtrn/realestate10k-130/videos/00001_000db54a47bd43fe.mp4', 'pose_path': '/kaggle/input/datasets/nhtdngtrn/realestate10k-130/poses/00001_000db54a47bd43fe.txt', 'prompt': 'A realistic real estate video with smooth camera movement.'}
{'id': '00002_000eb6240f06dd5a', 'video_path': '/kaggle/input/datasets/nhtdngtrn/realestate10k-130/videos/00002_000eb6240f06d

## Stage 2.2 — Video / Pose Utilities


In [13]:
def read_pose_file(path):
    lines = [ln.strip() for ln in Path(path).read_text(encoding="utf-8", errors="ignore").splitlines() if ln.strip()]
    if len(lines) < 2:
        raise ValueError(f"Pose file too short: {path}")
    url = lines[0]
    rows = []
    for ln in lines[1:]:
        parts = ln.split()
        if len(parts) < 17:
            continue
        ts = int(float(parts[0]))
        intr = np.array([float(x) for x in parts[1:5]], dtype=np.float32)
        pose_vals = np.array([float(x) for x in parts[5:]], dtype=np.float32)
        pose_3x4 = pose_vals[:12].reshape(3, 4) if len(pose_vals) >= 12 else np.zeros((3, 4), dtype=np.float32)
        pose_4x4 = np.eye(4, dtype=np.float32)
        pose_4x4[:3, :4] = pose_3x4
        rows.append({"timestamp": ts, "intrinsics": intr, "pose": pose_3x4, "pose4x4": pose_4x4, "raw_pose": pose_vals})
    if not rows:
        raise ValueError(f"No valid pose rows: {path}")
    return url, rows

def read_video_frame_at(cap, sec, height, width):
    cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, sec) * 1000.0)
    ok, frame = cap.read()
    if not ok or frame is None:
        return None
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    frame = cv2.resize(frame, (width, height), interpolation=cv2.INTER_AREA)
    return frame

def load_clip_from_video(video_path, pose_rows, total_frames, height, width):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")
    if len(pose_rows) >= total_frames:
        idxs = np.linspace(0, len(pose_rows) - 1, total_frames).round().astype(int)
        secs = [pose_rows[i]["timestamp"] / 1_000_000.0 for i in idxs]
        selected_poses = [pose_rows[i] for i in idxs]
    else:
        fps = cap.get(cv2.CAP_PROP_FPS) or 24
        frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT) or total_frames
        duration = frame_count / fps
        secs = np.linspace(0, max(0.0, duration - 1e-3), total_frames).tolist()
        idxs = np.linspace(0, len(pose_rows) - 1, total_frames).round().astype(int)
        selected_poses = [pose_rows[i] for i in idxs]
    frames = []
    last = None
    for sec in secs:
        fr = read_video_frame_at(cap, sec, height, width)
        if fr is None:
            fr = last if last is not None else np.zeros((height, width, 3), dtype=np.uint8)
        frames.append(fr)
        last = fr
    cap.release()
    return np.stack(frames), selected_poses

def to_tensor_video(frames):
    arr = frames.astype(np.float32) / 127.5 - 1.0
    arr = np.transpose(arr, (0, 3, 1, 2))
    return torch.from_numpy(arr)

def to_tensor_mask(masks):
    # T,H,W uint8/bool -> T,1,H,W float [0,1]
    arr = masks.astype(np.float32)
    if arr.max() > 1.0:
        arr = arr / 255.0
    arr = arr[:, None]
    return torch.from_numpy(arr)

def intrinsics_to_K(intr, height, width):
    fx, fy, cx, cy = [float(x) for x in intr]
    # RealEstate often stores normalized intrinsics.
    if fx < 10:
        fx *= width
    if fy < 10:
        fy *= height
    if cx <= 2:
        cx *= width
    if cy <= 2:
        cy *= height
    return np.array([[fx, 0, cx], [0, fy, cy], [0, 0, 1]], dtype=np.float32)

## Stage 2.3 — Keye-VL Prompt / Entity Helper


In [14]:
_KEYE = {"model": None, "processor": None, "loaded": False, "error": None}

def load_keye_once():
    if _KEYE["loaded"]:
        return _KEYE["model"], _KEYE["processor"]
    _KEYE["loaded"] = True
    try:
        from transformers import AutoModel, AutoProcessor
        model_path = str(KEYE_MODEL) if KEYE_MODEL is not None else "Kwai-Keye/Keye-VL-1_5-8B"
        print("Loading Keye-VL from", model_path)
        model = AutoModel.from_pretrained(
            model_path,
            trust_remote_code=True,
            torch_dtype="auto",
            local_files_only=(KEYE_MODEL is not None),
            device_map="auto" if DEVICE.type == "cuda" else None,
        ).eval()
        processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True, local_files_only=(KEYE_MODEL is not None))
        _KEYE["model"] = model
        _KEYE["processor"] = processor
        return model, processor
    except Exception as e:
        _KEYE["error"] = f"{type(e).__name__}: {e}"
        print("WARN Keye load failed:", _KEYE["error"])
        if STRICT_EXTERNAL_MODELS:
            raise
        return None, None

def keye_describe_video(video_path, fallback_prompt):
    if not RUN_KEYE:
        return {"prompt": DEFAULT_KEYE_PROMPT, "entities": ["moving objects"], "source": "fallback_disabled"}
    model, processor = load_keye_once()
    if model is None or processor is None:
        return {"prompt": fallback_prompt, "entities": ["moving objects"], "source": "fallback_load_failed", "error": _KEYE.get("error")}
    try:
        from keye_vl_utils import process_vision_info
        messages = [{
            "role": "user",
            "content": [
                {"type": "video", "video": str(video_path)},
                {"type": "text", "text": "Describe this real-estate video in one concise generation prompt, then list dynamic entities as comma-separated nouns. /no_think"},
            ],
        }]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(text=[text], images=image_inputs, videos=video_inputs, padding=True, return_tensors="pt")
        inputs = inputs.to(next(model.parameters()).device)
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_new_tokens=96, do_sample=False)
        out = processor.batch_decode(out_ids, skip_special_tokens=True)[0]
        # Very simple parser; keep full text for traceability.
        entities = []
        for part in re.split(r"[,;\n]", out):
            s = part.strip().lower()
            if 1 <= len(s.split()) <= 5 and any(w in s for w in ["person", "car", "animal", "object", "door", "curtain", "tree", "moving"]):
                entities.append(s)
        entities = entities[:5] or ["moving objects"]
        return {"prompt": out[:400] if out else fallback_prompt, "entities": entities, "source": "keye"}
    except Exception as e:
        print("WARN Keye inference failed:", type(e).__name__, e)
        if STRICT_EXTERNAL_MODELS:
            raise
        return {"prompt": fallback_prompt, "entities": ["moving objects"], "source": "fallback_infer_failed", "error": f"{type(e).__name__}: {e}"}

## Stage 2.4 — ReferDINO Dynamic Mask Helper


In [15]:

def optical_flow_motion_masks(frames, threshold_percentile=85):
    masks = []
    prev_gray = None
    for fr in frames:
        gray = cv2.cvtColor(fr, cv2.COLOR_RGB2GRAY)
        if prev_gray is None:
            masks.append(np.zeros(gray.shape, dtype=np.uint8))
        else:
            flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None, 0.5, 3, 15, 3, 5, 1.2, 0)
            mag = np.sqrt(flow[..., 0] ** 2 + flow[..., 1] ** 2)
            thr = np.percentile(mag, threshold_percentile)
            mask = (mag > max(thr, 0.15)).astype(np.uint8) * 255
            mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
            mask = cv2.dilate(mask, np.ones((5,5), np.uint8), iterations=1)
            masks.append(mask)
        prev_gray = gray
    return np.stack(masks)

class AttrDict(dict):
    """Tiny EasyDict fallback with recursive attribute access."""
    def __init__(self, *args, **kwargs):
        super().__init__()
        data = dict(*args, **kwargs)
        for k, v in data.items():
            self[k] = self._wrap(v)
    @staticmethod
    def _wrap(v):
        if isinstance(v, dict) and not isinstance(v, AttrDict):
            return AttrDict(v)
        if isinstance(v, list):
            return [AttrDict._wrap(x) for x in v]
        return v
    def __getattr__(self, name):
        try:
            return self[name]
        except KeyError:
            raise AttributeError(name)
    def __setattr__(self, name, value):
        self[name] = self._wrap(value)

def load_yaml_config(path):
    try:
        from ruamel.yaml import YAML
        with open(path, encoding="utf-8") as f:
            yaml = YAML(typ="safe", pure=True)
            return yaml.load(f)
    except Exception:
        import yaml
        with open(path, encoding="utf-8") as f:
            return yaml.safe_load(f)

def make_easydict(data):
    try:
        from easydict import EasyDict
        return EasyDict(data)
    except Exception:
        return AttrDict(data)

_REFERDINO = {"model": None, "args": None, "loaded": False, "error": None}

def load_referdino_once():
    if _REFERDINO["loaded"]:
        return _REFERDINO["model"], _REFERDINO["args"]
    _REFERDINO["loaded"] = True
    if not RUN_REFERDINO:
        _REFERDINO["error"] = "RUN_REFERDINO=False"
        return None, None
    if REFERDINO_REPO is None or REFERDINO_CKPT is None:
        _REFERDINO["error"] = "ReferDINO repo or checkpoint not found"
        return None, None
    try:
        repo = Path(REFERDINO_REPO)
        path_candidates = [
            repo,
            repo / "models" / "GroundingDINO",
            repo / "models" / "GroundingDINO" / "ops",
        ]
        for p in path_candidates:
            if p.exists() and str(p) not in sys.path:
                sys.path.insert(0, str(p))

        # If the build cell was skipped but the patched repo is available, try bootstrap once here.
        if not globals().get("REFERDINO_BUILD_OK", False) and (repo / "kaggle_bootstrap_referdino.py").exists():
            try:
                from kaggle_bootstrap_referdino import bootstrap
                built_repo = bootstrap(repo, jobs=2, verbose=True)
                globals()["REFERDINO_REPO"] = built_repo
                repo = Path(built_repo)
                for p in [repo, repo / "models" / "GroundingDINO", repo / "models" / "GroundingDINO" / "ops"]:
                    if str(p) not in sys.path:
                        sys.path.insert(0, str(p))
                globals()["REFERDINO_BUILD_OK"] = True
            except Exception as e:
                print("WARN late ReferDINO bootstrap failed:", type(e).__name__, e)
                if STRICT_EXTERNAL_MODELS:
                    raise

        config_path = repo / "configs" / "ytvos_swinb.yaml"
        if not config_path.exists():
            yamls = sorted(repo.rglob("*.yaml"))
            if not yamls:
                raise FileNotFoundError("No ReferDINO yaml config found")
            config_path = yamls[0]

        config = load_yaml_config(config_path)
        config = {k: v.get("value", v) if isinstance(v, dict) else v for k, v in config.items()}
        args = make_easydict({**config,
            "checkpoint_path": str(REFERDINO_CKPT),
            "device": "cuda" if DEVICE.type == "cuda" else "cpu",
            "enable_amp": DEVICE.type == "cuda",
            "tracking_alpha": 0.1,
        })
        if hasattr(args, "GroundingDINO"):
            args.GroundingDINO.tracking_alpha = args.tracking_alpha

        from models import build_model
        model, _, _ = build_model(args)
        model.to(args.device)
        checkpoint = torch.load(str(REFERDINO_CKPT), map_location="cpu")
        state_dict = checkpoint.get("model_state_dict", checkpoint.get("model", checkpoint))
        missing, unexpected = model.load_state_dict(state_dict, strict=False)
        print(f"ReferDINO state loaded. missing={len(missing)}, unexpected={len(unexpected)}")
        model.eval()
        _REFERDINO["model"] = model
        _REFERDINO["args"] = args
        print("ReferDINO loaded:", REFERDINO_CKPT)
        return model, args
    except Exception as e:
        _REFERDINO["error"] = f"{type(e).__name__}: {e}"
        print("WARN ReferDINO load failed:", _REFERDINO["error"])
        if STRICT_EXTERNAL_MODELS:
            raise
        return None, None

def referdino_masks_from_frames(frames, expression):
    if not RUN_REFERDINO:
        return optical_flow_motion_masks(frames), {"source": "optical_flow_disabled"}
    model, args = load_referdino_once()
    if model is None:
        return optical_flow_motion_masks(frames), {"source": "optical_flow_load_failed", "error": _REFERDINO.get("error")}
    try:
        import torchvision.transforms as T
        from misc import nested_tensor_from_videos_list
        transform = T.Compose([
            T.Resize(360),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        pil_frames = [Image.fromarray(fr) for fr in frames]
        imgs = torch.stack([transform(img) for img in pil_frames], dim=0).to(args.device)
        samples = nested_tensor_from_videos_list(imgs[None], size_divisibility=1)
        img_h, img_w = imgs.shape[-2:]
        target = {"size": torch.as_tensor([int(img_h), int(img_w)]).to(args.device)}
        exp = " ".join(str(expression).lower().split())
        with torch.no_grad():
            with torch.autocast(device_type="cuda", enabled=(args.device == "cuda")):
                outputs = model.infer(samples, [exp], [target])
        pred_logits = outputs["pred_logits"][0]
        pred_masks = outputs["pred_masks"][0]
        pred_scores = pred_logits.sigmoid().mean(0)
        max_scores, _ = pred_scores.max(-1)
        max_ind = max_scores.argmax(-1)
        video_len = len(frames)
        max_inds = max_ind.repeat(video_len)
        pred_masks = pred_masks[range(video_len), max_inds, ...].unsqueeze(0)
        pred_masks = pred_masks[:, :, :img_h, :img_w].cpu()
        origin_h, origin_w = frames[0].shape[:2]
        pred_masks = F.interpolate(pred_masks, size=(origin_h, origin_w), mode="bilinear", align_corners=False)
        masks = (pred_masks.sigmoid() > 0.5).squeeze(0).numpy().astype(np.uint8) * 255
        return masks, {"source": "referdino", "expression": exp}
    except Exception as e:
        print("WARN ReferDINO inference failed:", type(e).__name__, e)
        if STRICT_EXTERNAL_MODELS:
            raise
        return optical_flow_motion_masks(frames), {"source": "optical_flow_infer_failed", "error": f"{type(e).__name__}: {e}"}


## Stage 2.5 — MapAnything Geometry / Spatial Memory Helper


In [16]:
def normalize01(x):
    x = x.astype(np.float32)
    return (x - x.min()) / (x.max() - x.min() + 1e-6)

def fallback_depth_from_frames(frames):
    depths = []
    for fr in frames:
        gray = cv2.cvtColor(fr, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
        blur = cv2.GaussianBlur(gray, (0, 0), 3)
        edge = cv2.Laplacian(blur, cv2.CV_32F)
        depth = normalize01(1.0 - blur + 0.15 * np.abs(edge))
        depth = cv2.GaussianBlur(depth, (5, 5), 0)
        depths.append(depth.astype(np.float32))
    return np.stack(depths)

_MAPANYTHING = {"model": None, "loaded": False, "error": None}

def load_mapanything_once():
    if _MAPANYTHING["loaded"]:
        return _MAPANYTHING["model"]
    _MAPANYTHING["loaded"] = True
    if not RUN_MAPANYTHING:
        return None
    try:
        if MAPANYTHING_REPO is not None and str(MAPANYTHING_REPO) not in sys.path:
            sys.path.insert(0, str(MAPANYTHING_REPO))
        from mapanything.models import MapAnything
        model_path = str(MAPANYTHING_MODEL) if MAPANYTHING_MODEL is not None else "facebook/map-anything"
        print("Loading MapAnything from", model_path)
        model = MapAnything.from_pretrained(model_path, local_files_only=(MAPANYTHING_MODEL is not None)).to(DEVICE).eval()
        _MAPANYTHING["model"] = model
        return model
    except Exception as e:
        _MAPANYTHING["error"] = f"{type(e).__name__}: {e}"
        print("WARN MapAnything load failed:", _MAPANYTHING["error"])
        if STRICT_EXTERNAL_MODELS:
            raise
        return None

def mapanything_depth_and_points(frames, pose_rows):
    model = load_mapanything_once()
    if model is None:
        return fallback_depth_from_frames(frames), None, {"source": "fallback_depth", "error": _MAPANYTHING.get("error")}
    try:
        from mapanything.utils.image import preprocess_inputs
        views = []
        for fr, pose in zip(frames, pose_rows):
            K = intrinsics_to_K(pose["intrinsics"], fr.shape[0], fr.shape[1])
            views.append({
                "img": torch.from_numpy(fr).to(DEVICE),
                "intrinsics": torch.from_numpy(K).to(DEVICE),
                "camera_poses": torch.from_numpy(pose["pose4x4"]).to(DEVICE),
                "is_metric_scale": torch.tensor([True], device=DEVICE),
            })
        processed_views = preprocess_inputs(views)
        with torch.no_grad():
            predictions = model.infer(
                processed_views,
                memory_efficient_inference=True,
                minibatch_size=1,
                use_amp=(DEVICE.type == "cuda"),
                amp_dtype="bf16",
                apply_mask=True,
                mask_edges=True,
                apply_confidence_mask=False,
                ignore_calibration_inputs=False,
                ignore_pose_inputs=False,
            )
        depths = []
        pts_list = []
        for pred in predictions:
            d = pred.get("depth_z", pred.get("depth_along_ray"))
            if torch.is_tensor(d):
                d = d.detach().float().cpu().numpy()
            d = np.squeeze(d)
            if d.shape[:2] != frames[0].shape[:2]:
                d = cv2.resize(d, (frames[0].shape[1], frames[0].shape[0]), interpolation=cv2.INTER_LINEAR)
            depths.append(normalize01(d))
            pts = pred.get("pts3d")
            if torch.is_tensor(pts):
                pts = pts.detach().float().cpu().numpy()
            pts_list.append(pts)
        return np.stack(depths).astype(np.float32), pts_list, {"source": "mapanything"}
    except Exception as e:
        print("WARN MapAnything inference failed:", type(e).__name__, e)
        if STRICT_EXTERNAL_MODELS:
            raise
        return fallback_depth_from_frames(frames), None, {"source": "fallback_depth_infer_failed", "error": f"{type(e).__name__}: {e}"}

def render_depth_control(depths, masks=None):
    outs = []
    for i, depth in enumerate(depths):
        d8 = (normalize01(depth) * 255).astype(np.uint8)
        color = cv2.applyColorMap(d8, cv2.COLORMAP_TURBO)
        color = cv2.cvtColor(color, cv2.COLOR_BGR2RGB)
        if masks is not None:
            m = masks[min(i, len(masks)-1)] > 0
            color[m] = (0.35 * color[m] + np.array([255, 40, 40]) * 0.65).astype(np.uint8)
        outs.append(color)
    return np.stack(outs)

def render_pose_control(selected_poses, height, width):
    translations = np.array([p["pose"][:, 3] for p in selected_poses], dtype=np.float32)
    x = translations[:, 0]
    z = translations[:, 2]
    if np.ptp(x) < 1e-6:
        x = x + np.linspace(-0.01, 0.01, len(x))
    if np.ptp(z) < 1e-6:
        z = z + np.linspace(-0.01, 0.01, len(z))
    xs = ((x - x.min()) / (x.max() - x.min() + 1e-6) * (width * 0.8) + width * 0.1).astype(int)
    ys = ((z - z.min()) / (z.max() - z.min() + 1e-6) * (height * 0.8) + height * 0.1).astype(int)
    controls = []
    for i, p in enumerate(selected_poses):
        img = np.zeros((height, width, 3), dtype=np.uint8)
        for j in range(1, len(xs)):
            cv2.line(img, (xs[j-1], ys[j-1]), (xs[j], ys[j]), (40, 80, 160), 1)
        for j in range(1, i + 1):
            cv2.line(img, (xs[j-1], ys[j-1]), (xs[j], ys[j]), (60, 220, 80), 2)
        cv2.circle(img, (xs[i], ys[i]), 5, (255, 80, 60), -1)
        controls.append(img)
    return np.stack(controls)

def compose_memory_control(depth_control, pose_control):
    return np.clip(0.65 * depth_control.astype(np.float32) + 0.35 * pose_control.astype(np.float32), 0, 255).astype(np.uint8)

## Stage 2.6 — Reference Frame Retrieval Helper


In [17]:
def camera_overlap_score(pose_a, pose_b):
    ta = pose_a["pose"][:, 3]
    tb = pose_b["pose"][:, 3]
    dist = float(np.linalg.norm(ta - tb))
    ra = pose_a["pose"][:, :3]
    rb = pose_b["pose"][:, :3]
    rot_sim = float(np.trace(ra.T @ rb)) / 3.0
    return -dist + 0.1 * rot_sim

def select_reference_frames(candidate_frames, candidate_poses, target_poses, ref_frames=2):
    if len(candidate_frames) == 0:
        return np.zeros((ref_frames, HEIGHT, WIDTH, 3), dtype=np.uint8), []
    scores = []
    for i, cp in enumerate(candidate_poses):
        s = max(camera_overlap_score(cp, tp) for tp in target_poses)
        scores.append((s, i))
    selected = [i for _, i in sorted(scores, reverse=True)[:ref_frames]]
    refs = [candidate_frames[i] for i in selected]
    while len(refs) < ref_frames:
        refs.append(refs[-1])
    return np.stack(refs), selected

## Stage 3.1 — Offline Patches and Full Spatial Preprocess Cache


In [18]:
from pathlib import Path
import os, sys, re

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

def find_local_bert(root="/kaggle/input"):
    root = Path(root)
    hits = []

    for p in root.rglob("*"):
        if not p.is_dir():
            continue

        s = str(p).lower()
        has_config = (p / "config.json").exists()
        has_tokenizer = (p / "tokenizer.json").exists() or (p / "vocab.txt").exists()
        has_weight = (
            (p / "pytorch_model.bin").exists()
            or (p / "model.safetensors").exists()
            or any(p.glob("*.safetensors"))
        )

        if "bert" in s and has_config and has_tokenizer and has_weight:
            hits.append(p)

    return sorted(
        hits,
        key=lambda x: (
            0 if "bert-base-uncased" in str(x).lower() else 1,
            len(x.parts),
            str(x)
        )
    )

bert_hits = find_local_bert()

print("BERT candidates:")
for p in bert_hits[:20]:
    print(" -", p)

if not bert_hits:
    raise FileNotFoundError("Không tìm thấy BERT local trong /kaggle/input")

BERT_MODEL_DIR = bert_hits[0]
print("Using BERT_MODEL_DIR:", BERT_MODEL_DIR)

# Dùng repo ReferDINO đã build trong /kaggle/working
REFERDINO_REPO = Path("/kaggle/working/external_repos/ReferDINO-main")

if not REFERDINO_REPO.exists():
    raise FileNotFoundError("Không thấy ReferDINO repo đã build ở /kaggle/working/external_repos/ReferDINO-main")

# Quan trọng: thay NGUYÊN DÒNG text_encoder_type, không replace từng substring
for yml in REFERDINO_REPO.rglob("*.yaml"):
    txt = yml.read_text(encoding="utf-8")

    if "text_encoder_type" in txt:
        new_txt = re.sub(
            r'(^\s*text_encoder_type\s*:\s*).*$',
            rf'\1"{BERT_MODEL_DIR}"',
            txt,
            flags=re.MULTILINE
        )

        if new_txt != txt:
            yml.write_text(new_txt, encoding="utf-8")
            print("Patched:", yml)

# Add paths lại cho chắc
for p in [
    REFERDINO_REPO,
    REFERDINO_REPO / "models" / "GroundingDINO",
    REFERDINO_REPO / "models" / "GroundingDINO" / "ops",
]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        print("Added path:", p)

# Reset loader để ReferDINO load lại config mới
_REFERDINO = {"model": None, "args": None, "loaded": False, "error": None}

print("Done. Fixed BERT path safely.")
print("REFERDINO_REPO:", REFERDINO_REPO)
print("BERT_MODEL_DIR:", BERT_MODEL_DIR)

BERT candidates:
 - /kaggle/input/datasets/nhtdngtrn/bert-base-uncased-local
Using BERT_MODEL_DIR: /kaggle/input/datasets/nhtdngtrn/bert-base-uncased-local
Patched: /kaggle/working/external_repos/ReferDINO-main/configs/a2d_swint.yaml
Patched: /kaggle/working/external_repos/ReferDINO-main/configs/mevis_swint.yaml
Patched: /kaggle/working/external_repos/ReferDINO-main/configs/coco_swint.yaml
Patched: /kaggle/working/external_repos/ReferDINO-main/configs/a2d_swinb.yaml
Patched: /kaggle/working/external_repos/ReferDINO-main/configs/ytvos_swint.yaml
Patched: /kaggle/working/external_repos/ReferDINO-main/configs/davis_swinb.yaml
Patched: /kaggle/working/external_repos/ReferDINO-main/configs/jhmdb_swint.yaml
Patched: /kaggle/working/external_repos/ReferDINO-main/configs/coco_swinb.yaml
Patched: /kaggle/working/external_repos/ReferDINO-main/configs/davis_swint.yaml
Patched: /kaggle/working/external_repos/ReferDINO-main/configs/ytvos_swinb.yaml
Patched: /kaggle/working/external_repos/ReferDINO-

In [19]:
import os, sys, shutil, importlib.util

UNICEPTION_SRC = "/kaggle/input/datasets/nhtdngtrn/uniception"
UNICEPTION_DST = "/kaggle/working/UniCeption"

print("Source exists:", os.path.exists(UNICEPTION_SRC))
print("Source files:", os.listdir(UNICEPTION_SRC)[:10])

# Copy ra /kaggle/working
if os.path.exists(UNICEPTION_DST):
    shutil.rmtree(UNICEPTION_DST)

shutil.copytree(UNICEPTION_SRC, UNICEPTION_DST)

# Thêm repo root vào Python path
sys.path.insert(0, UNICEPTION_DST)

# Kiểm tra import
spec = importlib.util.find_spec("uniception")
print("uniception spec:", spec)

if spec is None:
    raise ModuleNotFoundError("Vẫn chưa tìm thấy uniception sau khi add sys.path")

import uniception
print("Import OK:", uniception.__file__)

Source exists: True
Source files: ['tests', '.pre-commit-config.yaml', 'uniception', 'scripts', 'LICENSE', '.gitignore', 'examples', '.github', 'pyproject.toml', 'README.md']
uniception spec: ModuleSpec(name='uniception', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7be45386be90>, origin='/kaggle/working/UniCeption/uniception/__init__.py', submodule_search_locations=['/kaggle/working/UniCeption/uniception'])
Import OK: /kaggle/working/UniCeption/uniception/__init__.py


In [20]:
import os, glob, shutil

# Tìm thư mục torch cache trong Kaggle Models
candidates = glob.glob("/kaggle/input/models/**/torch", recursive=True)

print("Candidates:")
for c in candidates:
    print("-", c)

if not candidates:
    raise FileNotFoundError("Không tìm thấy folder torch trong /kaggle/input/models")

# Ưu tiên đúng model DINOv2
src = None
for c in candidates:
    if "torch-hub-dinov2-vitl14" in c:
        src = c
        break

if src is None:
    src = candidates[0]

dst = "/root/.cache/torch"

if os.path.exists(dst):
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Copied torch cache:")
print("from:", src)
print("to:", dst)

print("\nCheck hub folders:")
!find /root/.cache/torch/hub -maxdepth 2 -type d | head -30

print("\nCheck checkpoints:")
!find /root/.cache/torch/hub/checkpoints -maxdepth 1 -type f | head -30

Candidates:
- /kaggle/input/models/nhtdngtrn/torch-hub-dinov2-vitl14/pytorch/default/1/torch
- /kaggle/input/models/nhtdngtrn/lpips-offline/pytorch/default/1/torch
Copied torch cache:
from: /kaggle/input/models/nhtdngtrn/torch-hub-dinov2-vitl14/pytorch/default/1/torch
to: /root/.cache/torch

Check hub folders:
/root/.cache/torch/hub
/root/.cache/torch/hub/checkpoints
/root/.cache/torch/hub/facebookresearch_dinov2_main
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2
/root/.cache/torch/hub/facebookresearch_dinov2_main/scripts
/root/.cache/torch/hub/facebookresearch_dinov2_main/.github
/root/.cache/torch/hub/facebookresearch_dinov2_main/docs
/root/.cache/torch/hub/facebookresearch_dinov2_main/notebooks

Check checkpoints:
/root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


In [21]:
from pathlib import Path
import sys, subprocess, importlib.util, os, textwrap

def find_wheel_dir(root="/kaggle/input"):
    root = Path(root)
    candidates = []
    for p in root.rglob("*"):
        if p.is_dir() and any(p.glob("*.whl")):
            if "mapanything" in str(p).lower() or "wheels" in str(p).lower():
                candidates.append(p)
    candidates = sorted(candidates, key=lambda x: (len(x.parts), str(x)))
    return candidates[0] if candidates else None

WHEEL_DIR = find_wheel_dir()
print("WHEEL_DIR:", WHEEL_DIR)

if WHEEL_DIR is None:
    raise FileNotFoundError("Không tìm thấy wheelhouse trong /kaggle/input")

# Chỉ install các package CÓ trong wheelhouse, không đụng torch/cuda
packages = [
    "hydra-core",
    "jaxtyping",
    "trimesh",
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-index",
    "--no-deps",
    f"--find-links={WHEEL_DIR}",
] + packages)

print("Installed available lightweight MapAnything deps")

# Tạo offline fallback cho các package nhỏ bị thiếu
OFFLINE_PKGS = Path("/kaggle/working/offline_pkgs")
OFFLINE_PKGS.mkdir(parents=True, exist_ok=True)

if str(OFFLINE_PKGS) not in sys.path:
    sys.path.insert(0, str(OFFLINE_PKGS))

# loralib fallback cho ReferDINO
(OFFLINE_PKGS / "loralib.py").write_text(r'''
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class Linear(nn.Linear):
    def __init__(self, in_features, out_features, r=0, lora_alpha=1, lora_dropout=0.0,
                 fan_in_fan_out=False, merge_weights=True, **kwargs):
        super().__init__(in_features, out_features, **kwargs)
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = lora_alpha / r if r and r > 0 else 1
        self.lora_dropout = nn.Dropout(p=lora_dropout) if lora_dropout > 0 else nn.Identity()
        if r and r > 0:
            self.lora_A = nn.Parameter(torch.zeros((r, in_features)))
            self.lora_B = nn.Parameter(torch.zeros((out_features, r)))
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B)

    def forward(self, x):
        result = F.linear(x, self.weight, self.bias)
        if getattr(self, "r", 0) and self.r > 0:
            result = result + F.linear(F.linear(self.lora_dropout(x), self.lora_A), self.lora_B) * self.scaling
        return result

def mark_only_lora_as_trainable(model, bias="none"):
    for n, p in model.named_parameters():
        p.requires_grad = "lora_" in n

def lora_state_dict(model, bias="none"):
    return {k: v for k, v in model.state_dict().items() if "lora_" in k}
''', encoding="utf-8")

# addict fallback
(OFFLINE_PKGS / "addict.py").write_text(r'''
class Dict(dict):
    def __init__(self, *args, **kwargs):
        super().__init__()
        data = dict(*args, **kwargs)
        for k, v in data.items():
            self[k] = self._wrap(v)

    @classmethod
    def _wrap(cls, v):
        if isinstance(v, dict) and not isinstance(v, Dict):
            return cls(v)
        if isinstance(v, list):
            return [cls._wrap(x) for x in v]
        return v

    def __getattr__(self, name):
        try:
            return self[name]
        except KeyError:
            raise AttributeError(name)

    def __setattr__(self, name, value):
        self[name] = self._wrap(value)
''', encoding="utf-8")

# termcolor fallback
(OFFLINE_PKGS / "termcolor.py").write_text(r'''
def colored(text, color=None, on_color=None, attrs=None):
    return text
''', encoding="utf-8")

# colorlog fallback
(OFFLINE_PKGS / "colorlog.py").write_text(r'''
import logging
class ColoredFormatter(logging.Formatter):
    pass
''', encoding="utf-8")

# yapf fallback
yapf_dir = OFFLINE_PKGS / "yapf" / "yapflib"
yapf_dir.mkdir(parents=True, exist_ok=True)
(OFFLINE_PKGS / "yapf" / "__init__.py").write_text("", encoding="utf-8")
(yapf_dir / "__init__.py").write_text("", encoding="utf-8")
(yapf_dir / "yapf_api.py").write_text(r'''
def FormatCode(code, *args, **kwargs):
    return code, False
''', encoding="utf-8")

print("Offline shim packages created:", OFFLINE_PKGS)

# Check không làm hỏng torch
import torch
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)

# Check imports
import hydra
print("OK hydra:", hydra.__file__)

import jaxtyping
print("OK jaxtyping")

import trimesh
print("OK trimesh")

import loralib
print("OK loralib:", loralib.__file__)

# Check MapAnything
if MAPANYTHING_REPO is not None and str(MAPANYTHING_REPO) not in sys.path:
    sys.path.insert(0, str(MAPANYTHING_REPO))

import mapanything
print("OK mapanything:", mapanything.__file__)

RUN_MAPANYTHING = True
_MAPANYTHING = {"model": None, "loaded": False, "error": None}

print("RUN_MAPANYTHING =", RUN_MAPANYTHING)

WHEEL_DIR: /kaggle/input/datasets/nhtdngtrn/hydra-132-mapanything-wheels
Looking in links: /kaggle/input/datasets/nhtdngtrn/hydra-132-mapanything-wheels
Installed available lightweight MapAnything deps
Offline shim packages created: /kaggle/working/offline_pkgs
torch: 2.10.0+cu128
cuda: 12.8
OK hydra: /usr/local/lib/python3.12/dist-packages/hydra/__init__.py
OK jaxtyping
OK trimesh
OK loralib: /kaggle/working/offline_pkgs/loralib.py
OK mapanything: /kaggle/input/datasets/nhtdngtrn/map-anything/map-anything-main/mapanything/__init__.py
RUN_MAPANYTHING = True


In [22]:
import os, glob, shutil

# Tìm checkpoint trong Kaggle Model input
candidates = glob.glob(
    "/kaggle/input/models/**/groundingdino_swinb_cogcoor.pth",
    recursive=True
)

print("Candidates:")
for c in candidates:
    print("-", c)

if not candidates:
    raise FileNotFoundError("Không tìm thấy groundingdino_swinb_cogcoor.pth trong /kaggle/input/models")

src = candidates[0]

# ReferDINO đang tìm relative path: pretrained/groundingdino_swinb_cogcoor.pth
os.makedirs("/kaggle/working/pretrained", exist_ok=True)

dst = "/kaggle/working/pretrained/groundingdino_swinb_cogcoor.pth"
shutil.copy2(src, dst)

os.chdir("/kaggle/working")

print("Copied checkpoint:")
print("from:", src)
print("to:", dst)
print("cwd:", os.getcwd())
print("exists:", os.path.exists(dst))

Candidates:
- /kaggle/input/models/nhtdngtrn/groundingdino-swinb-cogcoor/pytorch/default/1/pretrained/groundingdino_swinb_cogcoor.pth
Copied checkpoint:
from: /kaggle/input/models/nhtdngtrn/groundingdino-swinb-cogcoor/pytorch/default/1/pretrained/groundingdino_swinb_cogcoor.pth
to: /kaggle/working/pretrained/groundingdino_swinb_cogcoor.pth
cwd: /kaggle/working
exists: True


In [23]:
from pathlib import Path
import sys, importlib, subprocess

# ===== ReferDINO/GroundingDINO compatibility patch WITHOUT monkey-patching torch.Tensor.to =====
# Fixes:
# 1) BertModel.get_head_mask removed in newer transformers
# 2) get_extended_attention_mask API mismatch in newer transformers
# 3) bool tensor subtraction removed in newer PyTorch
# 4) tokenized.to(device) / dtype=torch.device issue in GroundingDINO text tokenization

candidates = [
    Path("/kaggle/working/external_repos/ReferDINO-main"),
    Path("/kaggle/working/ReferDINO-main"),
]

for p in Path("/kaggle/working").rglob("bertwarper.py"):
    if "GroundingDINO" in str(p) and len(p.parents) >= 3:
        candidates.append(p.parents[2])

repo = None
seen = set()
for c in candidates:
    c = Path(c)
    if c in seen:
        continue
    seen.add(c)
    if (c / "models" / "GroundingDINO" / "bertwarper.py").exists():
        repo = c
        break

if repo is None:
    raise FileNotFoundError("Không tìm thấy ReferDINO repo trong /kaggle/working. Hãy chạy cell bootstrap/copy ReferDINO trước.")

print("ReferDINO repo:", repo)

bertwarper = repo / "models" / "GroundingDINO" / "bertwarper.py"
swin_transformer = repo / "models" / "GroundingDINO" / "backbone" / "swin_transformer.py"
gdino_utils = repo / "models" / "GroundingDINO" / "utils.py"
groundingdino = repo / "models" / "GroundingDINO" / "groundingdino.py"

# ===== 1. Patch BertModel.get_head_mask fallback =====
text = bertwarper.read_text(encoding="utf-8")
old = "        self.get_head_mask = bert_model.get_head_mask\n"
new = """        # Compatibility fix: newer transformers may not expose get_head_mask on BertModel.
        self.get_head_mask = getattr(bert_model, \"get_head_mask\", self._get_head_mask)

    def _get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        if head_mask is None:
            return [None] * num_hidden_layers

        if head_mask.dim() == 1:
            head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
            head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
        elif head_mask.dim() == 2:
            head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)

        head_mask = head_mask.to(dtype=self.embeddings.word_embeddings.weight.dtype)

        if is_attention_chunked:
            head_mask = head_mask.unsqueeze(-1)

        return head_mask
"""
if old in text and "def _get_head_mask" not in text:
    text = text.replace(old, new)
    print("Patched bertwarper.py get_head_mask")
else:
    print("bertwarper.py get_head_mask already patched or pattern not found")

# ===== 2. Patch get_extended_attention_mask API mismatch =====
text = text.replace(
    "        self.get_extended_attention_mask = bert_model.get_extended_attention_mask\n",
    "        self.get_extended_attention_mask = self._get_extended_attention_mask\n"
)
if "def _get_extended_attention_mask(" not in text:
    marker = "    def forward(\n"
    method = """    def _get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
        if device is None:
            device = attention_mask.device
        if dtype is None:
            dtype = self.embeddings.word_embeddings.weight.dtype

        if attention_mask.dim() == 3:
            extended_attention_mask = attention_mask[:, None, :, :]
        elif attention_mask.dim() == 2:
            extended_attention_mask = attention_mask[:, None, None, :]
        else:
            raise ValueError(f\"Wrong shape for attention_mask: {attention_mask.shape}\")

        extended_attention_mask = extended_attention_mask.to(device=device, dtype=dtype)
        extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(dtype).min
        return extended_attention_mask

"""
    text = text.replace(marker, method + marker)
    print("Patched bertwarper.py extended_attention_mask")
else:
    print("bertwarper.py extended_attention_mask already patched")
bertwarper.write_text(text, encoding="utf-8")

# ===== 3. Patch bool subtraction in Swin Transformer =====
text = swin_transformer.read_text(encoding="utf-8")
old = "        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)\n"
new = """        if mask_windows.dtype == torch.bool:
            mask_windows = mask_windows.to(torch.int32)
        attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
"""
if old in text and "mask_windows.dtype == torch.bool" not in text:
    swin_transformer.write_text(text.replace(old, new, 1), encoding="utf-8")
    print("Patched swin_transformer.py bool subtraction")
else:
    print("swin_transformer.py already patched or pattern not found")

# ===== 4. Patch bool subtraction in GroundingDINO/utils.py =====
text = gdino_utils.read_text(encoding="utf-8")
old = "    attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)\n"
new = """    if mask_windows.dtype == torch.bool:
        mask_windows = mask_windows.to(torch.int32)
    attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
"""
if old in text and "mask_windows.dtype == torch.bool" not in text:
    gdino_utils.write_text(text.replace(old, new, 1), encoding="utf-8")
    print("Patched GroundingDINO/utils.py bool subtraction")
else:
    print("GroundingDINO/utils.py already patched or pattern not found")

# ===== 5. Patch GroundingDINO tokenized.to(device) without global torch monkey-patch =====
text = groundingdino.read_text(encoding="utf-8")
old = """        tokenized = self.tokenizer(captions, padding=\"longest\", return_tensors=\"pt\").to(
            device
        )
"""
new = """        tokenized = self.tokenizer(captions, padding=\"longest\", return_tensors=\"pt\")
        for _k, _v in list(tokenized.items()):
            if hasattr(_v, \"to\"):
                tokenized[_k] = _v.to(device=device)
"""
if old in text:
    groundingdino.write_text(text.replace(old, new), encoding="utf-8")
    print("Patched groundingdino.py tokenized.to(device):", text.count(old), "occurrence(s)")
else:
    print("groundingdino.py tokenized.to(device) already patched or pattern not found")

# ===== 6. Unload old imported modules so patched source is used =====
removed = []
for name in list(sys.modules.keys()):
    if name == "models" or name.startswith("models.") or name == "misc" or name.startswith("util.") or name == "MultiScaleDeformableAttention":
        removed.append(name)
        del sys.modules[name]
print("Removed imported ReferDINO/GroundingDINO modules:", len(removed))

for p in [repo, repo / "models" / "GroundingDINO", repo / "models" / "GroundingDINO" / "ops"]:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))
        print("Added path:", p)

importlib.invalidate_caches()
if "_REFERDINO" in globals():
    _REFERDINO = {"model": None, "args": None, "loaded": False, "error": None}
    print("Reset _REFERDINO lazy loader")

subprocess.check_call([sys.executable, "-m", "py_compile", str(bertwarper), str(swin_transformer), str(gdino_utils), str(groundingdino)])
print("All ReferDINO compatibility source patches applied. No torch.Tensor.to monkey-patch used.")


ReferDINO repo: /kaggle/working/external_repos/ReferDINO-main
bertwarper.py get_head_mask already patched or pattern not found
Patched bertwarper.py extended_attention_mask
Patched swin_transformer.py bool subtraction
Patched GroundingDINO/utils.py bool subtraction
Patched groundingdino.py tokenized.to(device): 2 occurrence(s)
Removed imported ReferDINO/GroundingDINO modules: 24
Reset _REFERDINO lazy loader
All ReferDINO compatibility source patches applied. No torch.Tensor.to monkey-patch used.


/kaggle/working/external_repos/ReferDINO-main/models/GroundingDINO/utils.py:66: SyntaxWarning: invalid escape sequence '\s'
  - memory: bs, \sum{hw}, d_model


In [24]:
# Disabled: no global torch.Tensor.to patch is used anymore.
# This avoids RecursionError affecting both ReferDINO and MapAnything.
import torch
print("torch.Tensor.to left unchanged:", torch.Tensor.to)


torch.Tensor.to left unchanged: <method 'to' of 'torch._C.TensorBase' objects>


In [25]:
from pathlib import Path
import sys, importlib, subprocess

# ===== Patch ReferDINO BertWarper for newer transformers =====
repo = Path("/kaggle/working/external_repos/ReferDINO-main")

if not repo.exists():
    # fallback auto-find
    hits = list(Path("/kaggle/working").rglob("models/GroundingDINO/bertwarper.py"))
    if not hits:
        raise FileNotFoundError("Không tìm thấy bertwarper.py trong /kaggle/working")
    repo = hits[0].parents[2]

bertwarper = repo / "models" / "GroundingDINO" / "bertwarper.py"
print("Patching:", bertwarper)

text = bertwarper.read_text(encoding="utf-8")

# 1. Không dùng trực tiếp HF get_extended_attention_mask nữa vì API transformers mới đổi
text = text.replace(
    "        self.get_extended_attention_mask = bert_model.get_extended_attention_mask\n",
    "        self.get_extended_attention_mask = self._get_extended_attention_mask\n"
)

# 2. Thêm hàm compatible nếu chưa có
if "def _get_extended_attention_mask(" not in text:
    marker = "    def forward(\n"
    method = '''    def _get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
        """Compatibility implementation for newer transformers versions.

        Old GroundingDINO passes `device` as the 3rd positional argument.
        Newer transformers may interpret that as `dtype`, causing:
        Tensor.to(dtype=torch.device(...)).
        """
        if device is None:
            device = attention_mask.device
        if dtype is None:
            dtype = self.embeddings.word_embeddings.weight.dtype

        if attention_mask.dim() == 3:
            extended_attention_mask = attention_mask[:, None, :, :]
        elif attention_mask.dim() == 2:
            extended_attention_mask = attention_mask[:, None, None, :]
        else:
            raise ValueError(
                f"Wrong shape for attention_mask: {attention_mask.shape}"
            )

        extended_attention_mask = extended_attention_mask.to(device=device, dtype=dtype)
        extended_attention_mask = (1.0 - extended_attention_mask) * torch.finfo(dtype).min
        return extended_attention_mask

'''
    text = text.replace(marker, method + marker)

bertwarper.write_text(text, encoding="utf-8")

# 3. Unload module cũ để Python import lại source mới
for name in list(sys.modules.keys()):
    if (
        name == "models"
        or name.startswith("models.")
        or name == "misc"
        or name.startswith("util.")
        or name == "MultiScaleDeformableAttention"
    ):
        del sys.modules[name]

importlib.invalidate_caches()

# 4. Reset ReferDINO loader
_REFERDINO = {"model": None, "args": None, "loaded": False, "error": None}

# 5. Add path lại
for p in [
    repo,
    repo / "models" / "GroundingDINO",
    repo / "models" / "GroundingDINO" / "ops",
]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

subprocess.check_call([sys.executable, "-m", "py_compile", str(bertwarper)])

print("Patched BertWarper extended_attention_mask. Now run Full preprocess again.")

Patching: /kaggle/working/external_repos/ReferDINO-main/models/GroundingDINO/bertwarper.py
Patched BertWarper extended_attention_mask. Now run Full preprocess again.


In [26]:
RUN_PREPROCESS = True

if RUN_PREPROCESS:
    for rec in tqdm(records, desc="Full preprocess"):
        out_path = PROC_DIR / f"{rec['id']}.pt"
        if out_path.exists():
            continue
        try:
            _, pose_rows = read_pose_file(rec["pose_path"])
            frames, selected_poses = load_clip_from_video(rec["video_path"], pose_rows, TOTAL_SAMPLE_FRAMES, HEIGHT, WIDTH)

            candidate_frames = frames[:CANDIDATE_FRAMES]
            prev_frames = frames[CANDIDATE_FRAMES:CANDIDATE_FRAMES + PREV_FRAMES]
            target_frames = frames[CANDIDATE_FRAMES + PREV_FRAMES:]
            candidate_poses = selected_poses[:CANDIDATE_FRAMES]
            prev_poses = selected_poses[CANDIDATE_FRAMES:CANDIDATE_FRAMES + PREV_FRAMES]
            target_poses = selected_poses[CANDIDATE_FRAMES + PREV_FRAMES:]

            keye_meta = keye_describe_video(rec["video_path"], rec["prompt"])
            entity_text = ", ".join(keye_meta.get("entities", ["moving objects"]))
            masks_all, mask_meta = referdino_masks_from_frames(frames, entity_text)
            target_masks = masks_all[CANDIDATE_FRAMES + PREV_FRAMES:]

            depth_all, pts3d, map_meta = mapanything_depth_and_points(frames, selected_poses)
            target_depth = depth_all[CANDIDATE_FRAMES + PREV_FRAMES:]
            depth_control = render_depth_control(target_depth, target_masks)
            pose_control = render_pose_control(target_poses, HEIGHT, WIDTH)
            memory_control = compose_memory_control(depth_control, pose_control)

            refs, ref_indices = select_reference_frames(candidate_frames, candidate_poses, target_poses, REF_FRAMES)

            sample = {
                "id": rec["id"],
                "prev": to_tensor_video(prev_frames),
                "target": to_tensor_video(target_frames),
                "control": to_tensor_video(depth_control),
                "memory": to_tensor_video(memory_control),
                "dynamic_mask": to_tensor_mask(target_masks),
                "reference": to_tensor_video(refs),
                "prompt": keye_meta.get("prompt", rec["prompt"]),
                "entities": keye_meta.get("entities", ["moving objects"]),
                "video_path": rec["video_path"],
                "pose_path": rec["pose_path"],
                "module_meta": {
                    "keye": keye_meta,
                    "referdino": mask_meta,
                    "mapanything": map_meta,
                    "reference_indices": ref_indices,
                },
            }
            torch.save(sample, out_path)
        except Exception as e:
            print("WARN preprocess failed:", rec["id"], type(e).__name__, e)
            if STRICT_EXTERNAL_MODELS:
                raise

processed_files = sorted(PROC_DIR.glob("*.pt"))
print("Processed samples:", len(processed_files))
print("Example:", processed_files[:3])
if len(processed_files) < 5:
    raise RuntimeError("Too few processed samples. Check video/pose files and module errors above.")

# Free heavy preprocess models before training.
try:
    _KEYE["model"] = None
    _MAPANYTHING["model"] = None
    _REFERDINO["model"] = None
except NameError:
    pass
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

Full preprocess:   0%|          | 0/120 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


final text_encoder_type: /kaggle/input/datasets/nhtdngtrn/bert-base-uncased-local
load tokenizer done.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/datasets/nhtdngtrn/bert-base-uncased-local
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading MapAnything from /kaggle/input/models/nguynhunhtrngc/map-anything-v1/pytorch/default/1/Map-anything-v1
Loading pretrained dinov2_vitl14 from torch hub


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


Loading weights from local directory


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
/kaggle/working/external_repos/ReferDINO-main/models/GroundingDINO/transformer.py:902: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1

Processed samples: 120
Example: [PosixPath('/kaggle/working/processed_spatia_full_wan2_p9_t49_c16_r7_100vid_stable/00000_000c3ab189999a83.pt'), PosixPath('/kaggle/working/processed_spatia_full_wan2_p9_t49_c16_r7_100vid_stable/00001_000db54a47bd43fe.pt'), PosixPath('/kaggle/working/processed_spatia_full_wan2_p9_t49_c16_r7_100vid_stable/00002_000eb6240f06dd5a.pt')]


## Stage 3.2 — Inspect Processed Sample


In [27]:
# Inspect one processed sample before training.
import torch, glob

pt_files = sorted(glob.glob(str(PROC_DIR / "*.pt")))
print("processed files:", len(pt_files))

sample = torch.load(pt_files[0], map_location="cpu")
print("Keys trong sample:")
for k in sample.keys():
    print("-", k)

print("Chi tiết shape:")
for k, v in sample.items():
    if hasattr(v, "shape"):
        print(k, v.shape, v.dtype)
    else:
        print(k, type(v))

print("module_meta:")
print(sample.get("module_meta", {}))


processed files: 120
Keys trong sample:
- id
- prev
- target
- control
- memory
- dynamic_mask
- reference
- prompt
- entities
- video_path
- pose_path
- module_meta
Chi tiết shape:
id <class 'str'>
prev torch.Size([9, 3, 192, 320]) torch.float32
target torch.Size([49, 3, 192, 320]) torch.float32
control torch.Size([49, 3, 192, 320]) torch.float32
memory torch.Size([49, 3, 192, 320]) torch.float32
dynamic_mask torch.Size([49, 1, 192, 320]) torch.float32
reference torch.Size([7, 3, 192, 320]) torch.float32
prompt <class 'str'>
entities <class 'list'>
video_path <class 'str'>
pose_path <class 'str'>
module_meta <class 'dict'>
module_meta:
{'keye': {'prompt': 'A realistic real estate video with smooth camera movement.', 'entities': ['moving objects'], 'source': 'fallback_disabled'}, 'referdino': {'source': 'referdino', 'expression': 'moving objects'}, 'mapanything': {'source': 'mapanything'}, 'reference_indices': [15, 14, 13, 12, 11, 10, 9]}


## Stage 4.1 — Dataset and Dataloader


In [28]:
class SpatiaFullDataset(Dataset):
    def __init__(self, files):
        self.files = list(files)
        prompts = []
        for f in self.files:
            item = torch.load(f, map_location="cpu")
            prompts.append(item.get("prompt", DEFAULT_PROMPT))
        uniq = sorted(set(prompts))
        self.prompt_to_id = {p: i for i, p in enumerate(uniq)}
        self.id_to_prompt = {i: p for p, i in self.prompt_to_id.items()}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        item = torch.load(self.files[idx], map_location="cpu")
        prompt = item.get("prompt", DEFAULT_PROMPT)
        return {
            "prev": item["prev"].float(),
            "target": item["target"].float(),
            "control": item["control"].float(),
            "memory": item["memory"].float(),
            "dynamic_mask": item["dynamic_mask"].float(),
            "reference": item["reference"].float(),
            "prompt_id": torch.tensor(self.prompt_to_id.get(prompt, 0), dtype=torch.long),
            "prompt_text": prompt,
            "id": item.get("id", Path(self.files[idx]).stem),
            "module_meta": item.get("module_meta", {}),
        }


def collate_fn(batch):
    tensor_keys = ["prev", "target", "control", "memory", "dynamic_mask", "reference", "prompt_id"]
    out = {k: torch.stack([b[k] for b in batch]) for k in tensor_keys}
    out["prompt_text"] = [b["prompt_text"] for b in batch]
    out["id"] = [b["id"] for b in batch]
    out["module_meta"] = [b["module_meta"] for b in batch]
    return out


all_files = sorted(PROC_DIR.glob("*.pt"))
if len(all_files) < TRAIN_VIDEOS:
    raise RuntimeError(f"Need at least {TRAIN_VIDEOS} processed training samples, found {len(all_files)} in {PROC_DIR}")

train_files = all_files[:TRAIN_VIDEOS]

# Use a real holdout if extra clips exist. If not, use a tiny train-proxy validation set
# so the assignment's 100-video-only requirement does not break Stage 5.
holdout_files = all_files[TRAIN_VIDEOS:TRAIN_VIDEOS + TEST_VIDEOS]
if len(holdout_files) > 0:
    test_files = holdout_files
    VAL_IS_TRAIN_PROXY = False
else:
    proxy_n = min(5, len(train_files))
    test_files = train_files[:proxy_n]
    VAL_IS_TRAIN_PROXY = True
    print(f"WARN: no extra validation files found; using {proxy_n} training samples as validation proxy.")

train_ds = SpatiaFullDataset(train_files)
val_ds = SpatiaFullDataset(test_files)
val_ds.prompt_to_id = train_ds.prompt_to_id
val_ds.id_to_prompt = train_ds.id_to_prompt

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    collate_fn=collate_fn,
    persistent_workers=(NUM_WORKERS > 0),
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    collate_fn=collate_fn,
    persistent_workers=(NUM_WORKERS > 0),
)

print("Train:", len(train_ds), "Val:", len(val_ds), "Val proxy:", VAL_IS_TRAIN_PROXY, "Prompts:", len(train_ds.prompt_to_id))
b = next(iter(train_loader))
for k, v in b.items():
    if torch.is_tensor(v):
        print(k, v.shape, v.dtype)
    elif isinstance(v, list):
        print(k, type(v), v[:1])
    else:
        print(k, type(v))

# Strict meta check: no silent fallback for core spatial modules.
for j, meta in enumerate(b["module_meta"]):
    for module_name in ["referdino", "mapanything"]:
        src = str(meta.get(module_name, {}).get("source", ""))
        if any(x in src for x in ["fallback", "optical_flow", "disabled", "failed"]):
            raise RuntimeError(f"Strict module check failed on batch item {j}: {module_name} source={src}")
print("Strict preprocessed-module check OK for first train batch.")


Train: 100 Val: 20 Val proxy: False Prompts: 1
prev torch.Size([1, 9, 3, 192, 320]) torch.float32
target torch.Size([1, 49, 3, 192, 320]) torch.float32
control torch.Size([1, 49, 3, 192, 320]) torch.float32
memory torch.Size([1, 49, 3, 192, 320]) torch.float32
dynamic_mask torch.Size([1, 49, 1, 192, 320]) torch.float32
reference torch.Size([1, 7, 3, 192, 320]) torch.float32
prompt_id torch.Size([1]) torch.int64
prompt_text <class 'list'> ['A realistic real estate video with smooth camera movement.']
id <class 'list'> ['00108_0c9ea3bf67254e95']
module_meta <class 'list'> [{'keye': {'prompt': 'A realistic real estate video with smooth camera movement.', 'entities': ['moving objects'], 'source': 'fallback_disabled'}, 'referdino': {'source': 'referdino', 'expression': 'moving objects'}, 'mapanything': {'source': 'mapanything'}, 'reference_indices': [15, 14, 13, 12, 11, 10, 9]}]
Strict preprocessed-module check OK for first train batch.


## Stage 4.2 — Wan2.2-backed Spatia Latent Model

Khác với notebook cũ, cell này load **Wan2.2 Diffusers pipeline** và dùng transformer/VAE/text encoder của Wan trong training. Nếu Wan2.2 không load được, notebook dừng.


In [29]:
def move_batch(batch, device):
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.to(device, non_blocking=True)
        else:
            out[k] = v
    return out


def flow_matching_loss(model, batch):
    return model.training_loss(batch)


def apply_stage_module_modes(model):
    """Keep frozen Wan modules in eval mode; only the intended trainable branch is in train mode."""
    if getattr(model, "vae", None) is not None:
        model.vae.eval()
    if getattr(model, "text_encoder", None) is not None:
        model.text_encoder.eval()

    if getattr(model, "control", None) is not None:
        model.control.train(model.stage == "stage1")

    if getattr(model, "transformer", None) is not None:
        model.transformer.train(model.stage == "stage2")


def evaluate(model, loader, max_batches=None):
    """Validation that can never leave the model stuck in eval stage.

    Previous bug:
    - evaluate() set model.stage = "eval".
    - if reconstruction/validation raised before the restore lines, model.stage stayed "eval".
    - next Stage 2 train step then computed loss without LoRA grad, so loss.backward()
      failed with: element 0 of tensors does not require grad and does not have a grad_fn.
    """
    if max_batches is None:
        max_batches = EVAL_MAX_BATCHES

    old_stage = getattr(model, "stage", None)
    old_training = model.training
    losses, psnrs = [], []

    try:
        model.stage = "eval"
        model.eval()

        with torch.no_grad():
            for i, batch in enumerate(loader):
                if i >= max_batches:
                    break

                loss = flow_matching_loss(model, batch)
                if torch.isfinite(loss):
                    losses.append(float(loss.detach().cpu()))

                pred, target, _ = model.reconstruct_video(batch, t_value=0.2)
                mse = F.mse_loss((pred + 1) / 2, (target + 1) / 2).item()
                psnr = -10 * math.log10(max(mse, 1e-8))
                psnrs.append(psnr)

        val_loss = float(np.mean(losses)) if len(losses) else float("nan")
        psnr_proxy = float(np.mean(psnrs)) if len(psnrs) else float("nan")
        return {"val_loss": val_loss, "psnr_proxy": psnr_proxy}

    finally:
        if old_stage is not None:
            model.stage = old_stage
        model.train(old_training)
        apply_stage_module_modes(model)


def set_stage1_trainable(model):
    model.stage = "stage1"
    for p in model.parameters():
        p.requires_grad = False
    for p in model.control.parameters():
        p.requires_grad = True
    apply_stage_module_modes(model)


def set_stage2_trainable(model):
    model.stage = "stage2"
    for p in model.parameters():
        p.requires_grad = False

    # Freeze trained control branch during LoRA fine-tuning, matching paper's two-stage intent.
    for p in model.control.parameters():
        p.requires_grad = False

    lora_count = 0
    for n, p in model.transformer.named_parameters():
        if "lora" in n.lower():
            p.requires_grad = True
            lora_count += p.numel()

    if lora_count == 0:
        raise RuntimeError("No LoRA parameters found in Wan transformer for Stage 2.")

    apply_stage_module_modes(model)
    print("Stage2 LoRA trainable params:", lora_count)


def _make_grad_scaler(enabled):
    if not enabled:
        return None
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            return torch.amp.GradScaler("cuda", enabled=True)
        except TypeError:
            return torch.amp.GradScaler(enabled=True)
    return torch.cuda.amp.GradScaler(enabled=True)


def _lr_scale(step, max_steps):
    warmup_steps = max(1, int(max_steps * WARMUP_RATIO))
    if step <= warmup_steps:
        return max(MIN_LR_SCALE, step / warmup_steps)
    progress = (step - warmup_steps) / max(1, max_steps - warmup_steps)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MIN_LR_SCALE + (1.0 - MIN_LR_SCALE) * cosine


def _free_gb(path):
    total, used, free = shutil.disk_usage(str(path))
    return free / 1024**3


def _to_cpu_detached(obj):
    """Recursively move tensors to CPU before checkpoint serialization."""
    if torch.is_tensor(obj):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: _to_cpu_detached(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_to_cpu_detached(v) for v in obj]
    if isinstance(obj, tuple):
        return tuple(_to_cpu_detached(v) for v in obj)
    return obj


def keep_latest_checkpoints(ckpt_dir, pattern="*.pt", keep=2):
    """Keep only the newest checkpoint files matching pattern."""
    ckpt_dir = Path(ckpt_dir)
    keep = max(int(keep), 0)
    files = sorted(
        ckpt_dir.glob(pattern),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    for p in files[keep:]:
        try:
            print("Remove old checkpoint:", p.name)
            p.unlink()
        except Exception as e:
            print("WARN cannot remove checkpoint", p, type(e).__name__, e)


def cleanup_tmp_checkpoints(ckpt_dir):
    ckpt_dir = Path(ckpt_dir)
    for p in ckpt_dir.glob("*.pt.tmp"):
        try:
            print("Remove unfinished temp checkpoint:", p.name)
            p.unlink()
        except Exception as e:
            print("WARN cannot remove temp checkpoint", p, type(e).__name__, e)


def safe_torch_save(obj, path, min_free_gb=1.0):
    """Atomic torch.save with CPU tensors and temp-file cleanup."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    cleanup_tmp_checkpoints(path.parent)

    free_before = _free_gb(path.parent)
    print(f"Free disk before save: {free_before:.2f} GB")
    if free_before < min_free_gb:
        raise RuntimeError(
            f"Not enough disk space to save checkpoint. Free: {free_before:.2f} GB, "
            f"required: {min_free_gb:.2f} GB. Delete old outputs or reduce checkpoint frequency."
        )

    tmp_path = path.with_suffix(path.suffix + ".tmp")
    if tmp_path.exists():
        tmp_path.unlink()

    try:
        torch.save(_to_cpu_detached(obj), tmp_path)
        os.replace(tmp_path, path)
    except Exception:
        if tmp_path.exists():
            tmp_path.unlink()
        raise

    print("Saved", path)
    print(f"Free disk after save: {_free_gb(path.parent):.2f} GB")


def train_loop(stage_name, model, loader, max_steps, lr):
    trainable = [p for p in model.parameters() if p.requires_grad]
    n_trainable = sum(p.numel() for p in trainable)

    if n_trainable == 0:
        raise RuntimeError(f"{stage_name}: no trainable parameters")

    print(stage_name, "trainable params:", n_trainable)

    # Keep the intended training stage explicit. This prevents a failed validation
    # pass from leaving model.stage="eval" and silently disabling LoRA gradients.
    stage_name_l = stage_name.lower()
    train_stage = "stage2" if "stage2" in stage_name_l else ("stage1" if "stage1" in stage_name_l else getattr(model, "stage", "stage1"))
    model.stage = train_stage
    apply_stage_module_modes(model)

    opt = torch.optim.AdamW(
        trainable,
        lr=lr,
        weight_decay=OPTIM_WEIGHT_DECAY,
        eps=OPTIM_EPS,
        foreach=False,
    )

    scaler = _make_grad_scaler(USE_GRAD_SCALER)
    apply_stage_module_modes(model)
    iterator = iter(loader)
    running = []
    start = time.time()
    consecutive_bad = 0
    skipped_steps = 0
    lr_safety_factor = 1.0

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    grad_clip = GRAD_CLIP_STAGE2 if "stage2" in stage_name.lower() else GRAD_CLIP_STAGE1

    for step in range(1, max_steps + 1):
        # Re-assert training state every step. Cheap and robust.
        model.stage = train_stage
        apply_stage_module_modes(model)

        current_lr = lr * _lr_scale(step, max_steps) * lr_safety_factor
        for group in opt.param_groups:
            group["lr"] = current_lr

        opt.zero_grad(set_to_none=True)
        total = 0.0
        bad_step = False

        for _ in range(GRAD_ACCUM_STEPS):
            try:
                batch = next(iterator)
            except StopIteration:
                iterator = iter(loader)
                batch = next(iterator)

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=AMP_DTYPE,
                enabled=(DEVICE.type == "cuda"),
            ):
                loss = flow_matching_loss(model, batch) / GRAD_ACCUM_STEPS

            if not torch.isfinite(loss):
                val = float(loss.detach().cpu()) if loss.numel() == 1 else loss
                print(f"[{stage_name}] step {step}: non-finite loss, skip optimizer step:", val)
                bad_step = True
                break

            if not loss.requires_grad:
                trainable_now = sum(p.numel() for p in model.parameters() if p.requires_grad)
                lora_trainable_now = sum(p.numel() for n, p in model.transformer.named_parameters() if "lora" in n.lower() and p.requires_grad)
                raise RuntimeError(
                    f"{stage_name}: loss has no grad_fn. "
                    f"model.stage={getattr(model, 'stage', None)!r}, "
                    f"trainable_now={trainable_now}, lora_trainable_now={lora_trainable_now}. "
                    "This usually means validation or an external cell left the model in eval/non-train stage."
                )

            if scaler is not None:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            total += float(loss.detach().cpu()) * GRAD_ACCUM_STEPS

        if bad_step:
            skipped_steps += 1
            consecutive_bad += 1
            opt.zero_grad(set_to_none=True)
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            if consecutive_bad % 3 == 0:
                lr_safety_factor = max(lr_safety_factor * 0.5, 0.05)
                print(f"[{stage_name}] LR backoff: safety_factor={lr_safety_factor:.3f}")
            if consecutive_bad >= MAX_CONSECUTIVE_BAD_STEPS:
                print(f"[{stage_name}] WARN: {consecutive_bad} consecutive bad steps; continuing with minimum LR backoff.")
                consecutive_bad = 0
            continue

        if scaler is not None:
            scaler.unscale_(opt)

        grad_norm = torch.nn.utils.clip_grad_norm_(trainable, grad_clip)

        if not torch.isfinite(grad_norm):
            skipped_steps += 1
            consecutive_bad += 1
            print(f"[{stage_name}] step {step}: non-finite grad norm, skip optimizer step")
            opt.zero_grad(set_to_none=True)
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()
            if consecutive_bad % 3 == 0:
                lr_safety_factor = max(lr_safety_factor * 0.5, 0.05)
                print(f"[{stage_name}] LR backoff: safety_factor={lr_safety_factor:.3f}")
            continue

        if scaler is not None:
            scaler.step(opt)
            scaler.update()
        else:
            opt.step()

        opt.zero_grad(set_to_none=True)
        consecutive_bad = 0
        running.append(total)

        if step % LOG_EVERY == 0 or step == 1:
            loss_msg = np.mean(running[-LOG_EVERY:]) if running else float("nan")
            msg = f"[{stage_name}] step {step}/{max_steps} loss={loss_msg:.4f}"
            msg += f" grad_norm={float(grad_norm):.4f}"
            msg += f" lr={current_lr:.2e}"
            msg += f" skipped={skipped_steps}"
            if DEVICE.type == "cuda":
                msg += f" peak_vram={torch.cuda.max_memory_allocated()/1024**3:.2f}GB"
            print(msg)

        if step % VAL_EVERY == 0 or step == max_steps:
            try:
                print("VAL", evaluate(model, val_loader))
            except Exception as e:
                print("WARN: validation skipped; training state restored:", type(e).__name__, e)
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()
            finally:
                model.stage = train_stage
                apply_stage_module_modes(model)

        if step % SAVE_EVERY == 0 or step == max_steps:
            pattern = f"{stage_name}_step_*_wan_trainable.pt"

            # Keep one previous backup before writing the new checkpoint.
            # This frees disk while still preserving a fallback if the new save fails.
            keep_latest_checkpoints(
                CKPT_DIR,
                pattern=pattern,
                keep=max(KEEP_LAST_CKPTS - 1, 1),
            )

            ckpt = CKPT_DIR / f"{stage_name}_step_{step}_wan_trainable.pt"
            payload = {
                "stage": stage_name,
                "step": step,
                "trainable_state": model.export_trainable_state(),
                "prompt_to_id": train_ds.prompt_to_id,
                "note": "Wan2.2-backed Spatia-style latent control + Wan transformer LoRA. Stores trainable delta only, not full Wan2.2 weights.",
                "skipped_steps": skipped_steps,
                "amp_dtype": str(AMP_DTYPE),
                "lr_safety_factor": lr_safety_factor,
            }
            safe_torch_save(payload, ckpt, min_free_gb=MIN_FREE_GB_FOR_SAVE)

            # Final cleanup: only keep the newest 1-2 checkpoints for this stage.
            keep_latest_checkpoints(
                CKPT_DIR,
                pattern=pattern,
                keep=KEEP_LAST_CKPTS,
            )
            if DEVICE.type == "cuda":
                torch.cuda.empty_cache()

    print(stage_name, "done in", round((time.time() - start) / 60, 2), "min", "skipped_steps:", skipped_steps)


In [30]:
# ============================================================
# Wan2.2-backed latent model with Spatia-style control branch
# ============================================================

try:
    import diffusers
    from diffusers import DiffusionPipeline
    print("diffusers:", diffusers.__version__)
except Exception as e:
    raise RuntimeError(
        "diffusers is required for Wan2.2 backbone training. "
        "Add a compatible diffusers wheel/dataset to Kaggle or use an environment that has it."
    ) from e

try:
    from peft import LoraConfig
    _HAS_PEFT = True
    print("peft available")
except Exception as e:
    _HAS_PEFT = False
    print("WARN peft not available:", type(e).__name__, e)
    if STRICT_WAN_BACKBONE:
        raise RuntimeError("peft is required for Stage 2 LoRA on Wan2.2 transformer.") from e


def first_parameter_device_dtype(module):
    p = next(module.parameters())
    return p.device, p.dtype


def count_module_params(module):
    return sum(p.numel() for p in module.parameters())


def extract_tensor_from_output(out):
    if torch.is_tensor(out):
        return out
    if isinstance(out, (tuple, list)):
        for x in out:
            if torch.is_tensor(x):
                return x
    for attr in ["sample", "pred", "prediction", "hidden_states", "last_hidden_state"]:
        if hasattr(out, attr):
            x = getattr(out, attr)
            if torch.is_tensor(x):
                return x
    raise TypeError(f"Cannot extract tensor from output type {type(out)}")


# Stable stronger control branch
# This replaces the old 3-layer Conv3D adapter.
# It is still much smaller than Wan2.2, but has explicit spatial/temporal residual modeling.
try:
    CONTROL_WIDTH
except NameError:
    CONTROL_WIDTH = 384
try:
    CONTROL_DEPTH
except NameError:
    CONTROL_DEPTH = 6
try:
    CONTROL_OUTPUT_SCALE
except NameError:
    CONTROL_OUTPUT_SCALE = 1.0


class SpatioTemporalResidualBlock(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.norm1 = nn.GroupNorm(32, width)
        self.temporal = nn.Conv3d(width, width, kernel_size=(3, 1, 1), padding=(1, 0, 0))

        self.norm2 = nn.GroupNorm(32, width)
        self.spatial = nn.Conv3d(width, width, kernel_size=(1, 3, 3), padding=(0, 1, 1))

        self.norm3 = nn.GroupNorm(32, width)
        self.ffn = nn.Sequential(
            nn.Conv3d(width, width * 4, kernel_size=1),
            nn.SiLU(),
            nn.Conv3d(width * 4, width, kernel_size=1),
        )

    def forward(self, x):
        x = x + self.temporal(F.silu(self.norm1(x)))
        x = x + self.spatial(F.silu(self.norm2(x)))
        x = x + self.ffn(F.silu(self.norm3(x)))
        return x


class LatentSpatiaControlNet(nn.Module):
    """
    Stable latent-space Spatia-style control branch.

    Inputs:
    - noisy Wan latent
    - encoded condition latent from prev/control/memory/reference
    - scalar flow timestep t

    Output:
    - residual with same shape as Wan transformer's velocity prediction.
    """

    def __init__(self, channels, width=CONTROL_WIDTH, depth=CONTROL_DEPTH):
        super().__init__()
        self.channels = channels
        self.width = width
        self.depth = depth

        self.input_proj = nn.Conv3d(channels * 2 + 1, width, kernel_size=3, padding=1)
        self.blocks = nn.ModuleList([SpatioTemporalResidualBlock(width) for _ in range(depth)])
        self.output_norm = nn.GroupNorm(32, width)
        self.output_proj = nn.Conv3d(width, channels, kernel_size=3, padding=1)

        # ControlNet-style safe start: initially output zero residual.
        nn.init.zeros_(self.output_proj.weight)
        nn.init.zeros_(self.output_proj.bias)

    def forward(self, noisy_latents, cond_latents, t):
        if cond_latents.shape[-3:] != noisy_latents.shape[-3:]:
            cond_latents = F.interpolate(
                cond_latents,
                size=noisy_latents.shape[-3:],
                mode="trilinear",
                align_corners=False,
            )

        if cond_latents.shape[1] != noisy_latents.shape[1]:
            c = noisy_latents.shape[1]
            if cond_latents.shape[1] < c:
                rep = math.ceil(c / cond_latents.shape[1])
                cond_latents = cond_latents.repeat(1, rep, 1, 1, 1)[:, :c]
            else:
                cond_latents = cond_latents[:, :c]

        if t.ndim == 0:
            t = t[None]

        t_map = t.float().view(-1, 1, 1, 1, 1)
        t_map = t_map.to(dtype=noisy_latents.dtype, device=noisy_latents.device)
        t_map = t_map.expand(
            noisy_latents.shape[0],
            1,
            noisy_latents.shape[2],
            noisy_latents.shape[3],
            noisy_latents.shape[4],
        )

        x = torch.cat([noisy_latents, cond_latents, t_map], dim=1)
        x = self.input_proj(x)

        for block in self.blocks:
            x = block(x)

        x = self.output_proj(F.silu(self.output_norm(x)))
        return x

class WanSpatiaTrainer(nn.Module):
    def __init__(self, wan_dir, lora_rank=64, lora_alpha=128, lora_dropout=0.0):
        super().__init__()
        self.wan_dir = Path(wan_dir)
        self.pipe = DiffusionPipeline.from_pretrained(
            self.wan_dir,
            torch_dtype=BACKBONE_DTYPE,
            local_files_only=True,
            low_cpu_mem_usage=True,
        )
        self.pipe.to(DEVICE)

        self.transformer = getattr(self.pipe, "transformer", None) or getattr(self.pipe, "unet", None)
        self.vae = getattr(self.pipe, "vae", None)
        self.text_encoder = getattr(self.pipe, "text_encoder", None)
        self.tokenizer = getattr(self.pipe, "tokenizer", None)
        self.scheduler = getattr(self.pipe, "scheduler", None)

        missing = [n for n in ["transformer", "vae", "text_encoder", "tokenizer", "scheduler"] if getattr(self, n) is None]
        if missing:
            raise RuntimeError(f"Wan pipeline missing required components: {missing}. Components: {self.pipe.components.keys()}")

        if ENABLE_GRADIENT_CHECKPOINTING:
            if hasattr(self.transformer, "enable_gradient_checkpointing"):
                self.transformer.enable_gradient_checkpointing()
                print("Transformer gradient checkpointing: enabled")
            elif hasattr(self.transformer, "gradient_checkpointing"):
                self.transformer.gradient_checkpointing = True
                print("Transformer gradient checkpointing flag: enabled")

        # Freeze full Wan backbone first.
        for module in [self.transformer, self.vae, self.text_encoder]:
            module.eval()
            for p in module.parameters():
                p.requires_grad = False

        if hasattr(self.vae, "enable_tiling"):
            self.vae.enable_tiling()
        if hasattr(self.vae, "enable_slicing"):
            self.vae.enable_slicing()

        self._vae_layout = None
        self.control = None
        self.control_channels = None
        self.stage = "stage1"
        self.lora_rank = lora_rank
        self.lora_alpha = lora_alpha
        self.lora_dropout = lora_dropout
        self.lora_target_modules = []

        self._inject_lora()

        wan_params = count_module_params(self.transformer) + count_module_params(self.vae) + count_module_params(self.text_encoder)
        print("Wan train pipeline class:", type(self.pipe).__name__)
        print("Transformer class:", type(self.transformer).__name__)
        print("VAE class:", type(self.vae).__name__)
        print("Text encoder class:", type(self.text_encoder).__name__)
        print("Approx Wan component params:", f"{wan_params/1e9:.3f}B")
        if STRICT_WAN_BACKBONE and wan_params < 1_000_000_000:
            raise RuntimeError(f"Wan backbone check failed: only {wan_params/1e6:.1f}M params detected.")

    def _inject_lora(self):
        if not _HAS_PEFT:
            return
        # Detect linear module suffixes inside attention/ffn blocks.
        linear_names = [n for n, m in self.transformer.named_modules() if isinstance(m, nn.Linear)]
        preferred_tails = [
            "to_q", "to_k", "to_v", "to_out.0",
            "add_q_proj", "add_k_proj", "add_v_proj", "to_add_out",
            "q", "k", "v", "o", "proj", "proj_out", "fc1", "fc2",
        ]
        tails_present = {n.split(".")[-1] for n in linear_names}
        targets = [t for t in preferred_tails if t.split(".")[-1] in tails_present or t in linear_names]
        if not targets:
            # Conservative fallback: target attention-related Linear suffixes.
            attn_names = [n for n in linear_names if "attn" in n.lower() or "attention" in n.lower()]
            targets = sorted({n.split(".")[-1] for n in attn_names})[:8]
        if not targets:
            raise RuntimeError("Could not infer LoRA target modules inside Wan transformer.")

        self.lora_target_modules = sorted(set(targets))
        config = LoraConfig(
            r=self.lora_rank,
            lora_alpha=self.lora_alpha,
            lora_dropout=self.lora_dropout,
            init_lora_weights=True,
            target_modules=self.lora_target_modules,
        )
        if hasattr(self.transformer, "add_adapter"):
            self.transformer.add_adapter(config)
        else:
            raise RuntimeError("Wan transformer does not support add_adapter; cannot attach PEFT LoRA.")
        for n, p in self.transformer.named_parameters():
            p.requires_grad = "lora" in n.lower()
        print("LoRA attached to Wan transformer target modules:", self.lora_target_modules)

    def _text_embeds(self, prompts):
        device, dtype = first_parameter_device_dtype(self.text_encoder)
        tok = self.tokenizer(
            prompts,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        tok = {k: v.to(device) for k, v in tok.items()}
        with torch.no_grad():
            out = self.text_encoder(**tok)
        emb = extract_tensor_from_output(out)
        return emb.to(device=DEVICE, dtype=dtype)

    def _vae_encode_raw(self, video, layout):
        if layout == "BCTHW":
            x = video.permute(0, 2, 1, 3, 4).contiguous()
        elif layout == "BTCHW":
            x = video.contiguous()
        else:
            raise ValueError(layout)
        _, dtype = first_parameter_device_dtype(self.vae)
        x = x.to(device=DEVICE, dtype=dtype)
        out = self.vae.encode(x)
        if hasattr(out, "latent_dist"):
            z = out.latent_dist.sample()
        else:
            z = extract_tensor_from_output(out)
        scale = float(getattr(getattr(self.vae, "config", object()), "scaling_factor", 1.0))
        return z * scale

    def _sanitize_latents(self, z, name="latents"):
        z = z.float()
        if not torch.isfinite(z).all():
            print(f"WARN: non-finite values detected in {name}; applying nan_to_num.")
            z = torch.nan_to_num(z, nan=0.0, posinf=LATENT_CLAMP_VALUE, neginf=-LATENT_CLAMP_VALUE)
        z = z.clamp(-LATENT_CLAMP_VALUE, LATENT_CLAMP_VALUE)
        return z.to(device=DEVICE)

    def encode_video(self, video):
        # video: [B,T,C,H,W], range [-1,1]
        with torch.no_grad():
            if self._vae_layout is not None:
                z = self._vae_encode_raw(video, self._vae_layout)
                return self._sanitize_latents(z, "vae_latents")
            errors = []
            for layout in ["BCTHW", "BTCHW"]:
                try:
                    z = self._vae_encode_raw(video, layout)
                    self._vae_layout = layout
                    z = self._sanitize_latents(z, "vae_latents")
                    print("VAE encode layout selected:", layout, "latent shape:", tuple(z.shape))
                    return z
                except Exception as e:
                    errors.append((layout, f"{type(e).__name__}: {e}"))
                    if DEVICE.type == "cuda":
                        torch.cuda.empty_cache()
            raise RuntimeError(f"Wan VAE encode failed for all layouts: {errors}")

    def decode_video(self, latents):
        scale = float(getattr(getattr(self.vae, "config", object()), "scaling_factor", 1.0))
        z = latents / scale
        _, dtype = first_parameter_device_dtype(self.vae)
        with torch.no_grad():
            out = self.vae.decode(z.to(device=DEVICE, dtype=dtype))
        x = extract_tensor_from_output(out)
        # Convert likely B,C,T,H,W to B,T,C,H,W.
        if x.ndim == 5 and x.shape[1] in [1, 3, 4, 8, 16] and x.shape[2] != 3:
            x = x.permute(0, 2, 1, 3, 4).contiguous()
        return x.float().clamp(-1, 1)

    def make_condition_video(self, batch):
        target = batch["target"]
        T = target.shape[1]
        prev = batch["prev"]
        control = batch["control"]
        memory = batch["memory"]
        refs = batch["reference"]
        prev_summary = prev.mean(dim=1, keepdim=True).repeat(1, T, 1, 1, 1)
        ref_summary = refs.mean(dim=1, keepdim=True).repeat(1, T, 1, 1, 1)
        cond = 0.30 * prev_summary + 0.30 * control + 0.25 * memory + 0.15 * ref_summary
        return cond.clamp(-1, 1)

    def initialize_control_from_batch(self, batch):
        batch = move_batch(batch, DEVICE)
        latents = self.encode_video(batch["target"])
        channels = latents.shape[1]
        self.control_channels = channels
        self.control = LatentSpatiaControlNet(
            channels=channels,
            width=CONTROL_WIDTH,
            depth=CONTROL_DEPTH,
        ).to(DEVICE, dtype=torch.float32)
        control_params = count_module_params(self.control)
        print("Latent control initialized. latent shape:", tuple(latents.shape), "channels:", channels)
        print("Control branch params:", control_params)
        return latents.shape

    def _transformer_call(self, noisy_latents, t, prompt_embeds, grad_enabled):
        tr = self.transformer
        sig = inspect.signature(tr.forward)

        # IMPORTANT dtype guard:
        # During training, autocast usually casts inputs to BF16 automatically.
        # During validation/reconstruction, there may be no active autocast, so
        # noisy_latents can remain FP32 while Wan transformer weights/bias are BF16.
        # That causes: Input type (float) and bias type (c10::BFloat16) should be the same.
        # Cast the transformer input explicitly to the backbone compute dtype.
        tr_dtype = BACKBONE_DTYPE
        try:
            for _n, _p in tr.named_parameters():
                if "lora" not in _n.lower():
                    tr_dtype = _p.dtype
                    break
        except Exception:
            pass

        noisy_latents = noisy_latents.to(device=DEVICE, dtype=tr_dtype).contiguous()
        prompt_embeds = prompt_embeds.to(device=DEVICE, dtype=tr_dtype)
        timestep = (t * 1000).to(device=DEVICE, dtype=torch.float32)

        candidates = {
            "hidden_states": noisy_latents,
            "sample": noisy_latents,
            "latents": noisy_latents,
            "x": noisy_latents,
            "timestep": timestep,
            "timesteps": timestep,
            "t": timestep,
            "encoder_hidden_states": prompt_embeds,
            "context": prompt_embeds,
            "return_dict": False,
        }
        kwargs = {k: v for k, v in candidates.items() if k in sig.parameters}
        if not any(k in kwargs for k in ["hidden_states", "sample", "latents", "x"]):
            # Most diffusers models accept hidden_states even if signature is generic.
            kwargs["hidden_states"] = noisy_latents
        if not any(k in kwargs for k in ["timestep", "timesteps", "t"]):
            kwargs["timestep"] = timestep
        if not any(k in kwargs for k in ["encoder_hidden_states", "context"]):
            kwargs["encoder_hidden_states"] = prompt_embeds
        with torch.set_grad_enabled(grad_enabled):
            out = tr(**kwargs)
        pred = extract_tensor_from_output(out)
        if pred.shape != noisy_latents.shape:
            # Try common case: tuple returns [B,T,C,H,W].
            if pred.ndim == 5 and pred.shape[1] == noisy_latents.shape[2] and pred.shape[2] == noisy_latents.shape[1]:
                pred = pred.permute(0, 2, 1, 3, 4).contiguous()
            if pred.shape != noisy_latents.shape:
                raise RuntimeError(f"Wan transformer output shape {tuple(pred.shape)} != latent shape {tuple(noisy_latents.shape)}")
        return pred

    def training_loss(self, batch):
        batch = move_batch(batch, DEVICE)
        prompts = batch.get("prompt_text", [DEFAULT_PROMPT] * batch["target"].shape[0])

        latents = self.encode_video(batch["target"]).float().clamp(-LATENT_CLAMP_VALUE, LATENT_CLAMP_VALUE)
        cond_latents = self.encode_video(self.make_condition_video(batch)).float().clamp(-LATENT_CLAMP_VALUE, LATENT_CLAMP_VALUE)

        if self.control is None:
            self.initialize_control_from_batch(batch)

        B = latents.shape[0]
        t = torch.empty(B, device=DEVICE, dtype=torch.float32).uniform_(TIMESTEP_MIN, TIMESTEP_MAX)
        expand = (slice(None),) + (None,) * (latents.ndim - 1)

        noise = torch.randn_like(latents).clamp(-NOISE_CLAMP_VALUE, NOISE_CLAMP_VALUE)
        noisy_latents = ((1 - t[expand]) * latents + t[expand] * noise).clamp(-LATENT_CLAMP_VALUE, LATENT_CLAMP_VALUE)
        target_v = (noise - latents).float().clamp(-PRED_CLAMP_VALUE, PRED_CLAMP_VALUE)

        prompt_embeds = self._text_embeds(prompts)
        train_lora = self.stage == "stage2"

        base_pred = self._transformer_call(noisy_latents, t.float(), prompt_embeds, grad_enabled=train_lora)
        if self.stage == "stage1":
            base_pred = base_pred.detach()

        base_pred = torch.nan_to_num(base_pred.float(), nan=0.0, posinf=PRED_CLAMP_VALUE, neginf=-PRED_CLAMP_VALUE)
        base_pred = base_pred.clamp(-PRED_CLAMP_VALUE, PRED_CLAMP_VALUE)

        with torch.autocast(device_type=DEVICE.type, enabled=False):
            control_pred = self.control(
                noisy_latents.float(),
                cond_latents.float(),
                t.float(),
            )

        control_pred = torch.nan_to_num(control_pred.float(), nan=0.0, posinf=PRED_CLAMP_VALUE, neginf=-PRED_CLAMP_VALUE)
        control_pred = torch.tanh(control_pred) * CONTROL_OUTPUT_SCALE

        pred = (base_pred + control_pred).clamp(-PRED_CLAMP_VALUE, PRED_CLAMP_VALUE)
        target_v = torch.nan_to_num(target_v, nan=0.0, posinf=PRED_CLAMP_VALUE, neginf=-PRED_CLAMP_VALUE)

        # Optional static weighting from dynamic mask downsampled to latent dimensions.
        mask = batch["dynamic_mask"].to(device=DEVICE, dtype=torch.float32)  # [B,T,1,H,W]
        mask = mask.permute(0, 2, 1, 3, 4)
        mask = F.interpolate(mask, size=pred.shape[-3:], mode="trilinear", align_corners=False)
        mask = torch.nan_to_num(mask, nan=0.0, posinf=1.0, neginf=0.0).clamp(0.0, 1.0)
        weight = CONTROL_LOSS_STATIC_WEIGHT * (1 - mask) + CONTROL_LOSS_DYNAMIC_WEIGHT * mask

        diff = (pred - target_v).clamp(-LOSS_DIFF_CLAMP_VALUE, LOSS_DIFF_CLAMP_VALUE)
        loss = (diff.square() * weight).mean()
        return torch.nan_to_num(loss, nan=0.0, posinf=LOSS_DIFF_CLAMP_VALUE ** 2, neginf=0.0)

    @torch.no_grad()
    def reconstruct_video(self, batch, t_value=0.5):
        batch = move_batch(batch, DEVICE)
        prompts = batch.get("prompt_text", [DEFAULT_PROMPT] * batch["target"].shape[0])
        latents = self.encode_video(batch["target"]).float().clamp(-LATENT_CLAMP_VALUE, LATENT_CLAMP_VALUE)
        cond_latents = self.encode_video(self.make_condition_video(batch)).float().clamp(-LATENT_CLAMP_VALUE, LATENT_CLAMP_VALUE)
        B = latents.shape[0]
        t_scalar = min(max(float(t_value), TIMESTEP_MIN), TIMESTEP_MAX)
        t = torch.full((B,), t_scalar, device=DEVICE, dtype=torch.float32)
        expand = (slice(None),) + (None,) * (latents.ndim - 1)
        noise = torch.randn_like(latents).clamp(-NOISE_CLAMP_VALUE, NOISE_CLAMP_VALUE)
        noisy_latents = ((1 - t[expand]) * latents + t[expand] * noise).clamp(-LATENT_CLAMP_VALUE, LATENT_CLAMP_VALUE)
        prompt_embeds = self._text_embeds(prompts)
        base_pred = self._transformer_call(noisy_latents, t.float(), prompt_embeds, grad_enabled=False)
        with torch.autocast(device_type=DEVICE.type, enabled=False):
            control_pred = self.control(
                noisy_latents.float(),
                cond_latents.float(),
                t.float(),
            )

        control_pred = torch.tanh(control_pred) * CONTROL_OUTPUT_SCALE
        pred_v = (base_pred.float() + control_pred).clamp(-PRED_CLAMP_VALUE, PRED_CLAMP_VALUE)
        pred_v = torch.nan_to_num(pred_v, nan=0.0, posinf=PRED_CLAMP_VALUE, neginf=-PRED_CLAMP_VALUE)
        latents_hat = (noisy_latents.float() - t[expand] * pred_v).clamp(-LATENT_CLAMP_VALUE, LATENT_CLAMP_VALUE)
        video_hat = self.decode_video(latents_hat)
        target_video = batch["target"].float().clamp(-1, 1)
        # If VAE decode resolution differs, resize target for metric consistency.
        if video_hat.shape[-2:] != target_video.shape[-2:]:
            B,T,C,H,W = target_video.shape
            target_video = F.interpolate(target_video.flatten(0,1), size=video_hat.shape[-2:], mode="bilinear", align_corners=False).view(B,T,C,*video_hat.shape[-2:])
        if video_hat.shape[1] != target_video.shape[1]:
            T = min(video_hat.shape[1], target_video.shape[1])
            video_hat = video_hat[:, :T]
            target_video = target_video[:, :T]
        return video_hat, target_video, batch

    def export_trainable_state(self):
        control_state = None
        if self.control is not None:
            control_state = {
                k: v.detach().cpu() if torch.is_tensor(v) else v
                for k, v in self.control.state_dict().items()
            }

        payload = {
            "control": control_state,
            "lora": {k: v.detach().cpu() for k, v in self.transformer.state_dict().items() if "lora" in k.lower()},
            "lora_target_modules": self.lora_target_modules,
            "wan_dir": str(self.wan_dir),
            "control_width": CONTROL_WIDTH,
            "control_depth": CONTROL_DEPTH,
            "control_output_scale": CONTROL_OUTPUT_SCALE,
            "control_channels": self.control_channels,
        }
        return payload


model = WanSpatiaTrainer(
    WAN_DIR,
    lora_rank=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
).to(DEVICE)

# Build latent control once using a real batch.
with torch.no_grad():
    b0 = next(iter(train_loader))
    latent_shape = model.initialize_control_from_batch(b0)

wan_total = count_module_params(model.transformer) + count_module_params(model.vae) + count_module_params(model.text_encoder)
trainable_now = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Wan total params approx:", f"{wan_total/1e9:.3f}B")
print("Current trainable params after init:", trainable_now)

if not USE_WAN2_BACKBONE or ALLOW_TOY_ADAPTER:
    raise RuntimeError("Config invalid: this strict notebook must use Wan2.2 and must not allow toy adapter fallback.")


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


diffusers: 0.37.1
peft available


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/242 [00:00<?, ?it/s]

UMT5EncoderModel LOAD REPORT from: /kaggle/input/models/nhtdngtrn/wan2-2-5b-diffuser/pytorch/default/1/text_encoder
Key                         | Status  | 
----------------------------+---------+-
encoder.embed_tokens.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The config attributes {'clip_output': False} were passed to AutoencoderKLWan, but are not expected and will be ignored. Please verify your config.json configuration file.


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Transformer gradient checkpointing: enabled
LoRA attached to Wan transformer target modules: ['proj', 'proj_out', 'to_k', 'to_out.0', 'to_q', 'to_v']
Wan train pipeline class: WanPipeline
Transformer class: WanTransformer3DModel
VAE class: AutoencoderKLWan
Text encoder class: UMT5EncoderModel
Approx Wan component params: 12.564B
VAE encode layout selected: BCTHW latent shape: (1, 48, 13, 12, 20)
Latent control initialized. latent shape: (1, 48, 13, 12, 20) channels: 48
Control branch params: 19229232
Wan total params approx: 12.564B
Current trainable params after init: 147233328


## Stage 4.3 — Stage 1 Training: Spatia Latent Control Branch

In [31]:
RUN_STAGE1 = True
if RUN_STAGE1:
    set_stage1_trainable(model)
    train_loop("stage1_wan_spatia_control", model, train_loader, MAX_TRAIN_STEPS_STAGE1, LR_STAGE1)
else:
    print("Skipping Stage 1")


stage1_wan_spatia_control trainable params: 19229232
[stage1_wan_spatia_control] step 1/800 loss=1.1364 grad_norm=0.4468 lr=5.00e-07 skipped=0 peak_vram=46.21GB
[stage1_wan_spatia_control] step 25/800 loss=1.9105 grad_norm=0.4530 lr=1.56e-06 skipped=0 peak_vram=46.21GB
[stage1_wan_spatia_control] step 50/800 loss=1.8813 grad_norm=0.6378 lr=3.13e-06 skipped=0 peak_vram=46.21GB
[stage1_wan_spatia_control] step 75/800 loss=1.6942 grad_norm=0.5056 lr=4.69e-06 skipped=0 peak_vram=46.21GB
[stage1_wan_spatia_control] step 100/800 loss=1.7505 grad_norm=0.4062 lr=4.99e-06 skipped=0 peak_vram=46.21GB
WARN: validation skipped; training state restored: RuntimeError Input type (float) and bias type (c10::BFloat16) should be the same
Free disk before save: 3.80 GB
Saved /kaggle/working/spatia_full_checkpoints_wan2_p9_t49_c16_r7_100vid_stable/stage1_wan_spatia_control_step_100_wan_trainable.pt
Free disk after save: 3.49 GB
[stage1_wan_spatia_control] step 125/800 loss=1.7462 grad_norm=0.2964 lr=4.96e

## Stage 4.4 — Stage 2 Training: Wan2.2 Transformer LoRA

In [32]:
RUN_STAGE2 = True
if RUN_STAGE2:
    set_stage2_trainable(model)
    train_loop("stage2_wan_lora", model, train_loader, MAX_TRAIN_STEPS_STAGE2, LR_STAGE2)
else:
    print("Skipping Stage 2")


Stage2 LoRA trainable params: 128004096
stage2_wan_lora trainable params: 128004096
[stage2_wan_lora] step 1/500 loss=1.8590 grad_norm=0.1602 lr=1.00e-07 skipped=0 peak_vram=26.21GB
[stage2_wan_lora] step 25/500 loss=1.4987 grad_norm=0.1475 lr=5.00e-07 skipped=0 peak_vram=26.69GB
[stage2_wan_lora] step 50/500 loss=1.4696 grad_norm=0.1865 lr=1.00e-06 skipped=0 peak_vram=26.69GB
[stage2_wan_lora] step 75/500 loss=1.5558 grad_norm=0.1387 lr=9.93e-07 skipped=0 peak_vram=26.69GB
[stage2_wan_lora] step 100/500 loss=1.7223 grad_norm=0.1182 lr=9.73e-07 skipped=0 peak_vram=26.69GB
WARN: validation skipped; training state restored: RuntimeError Input type (float) and bias type (c10::BFloat16) should be the same
Free disk before save: 3.49 GB
Saved /kaggle/working/spatia_full_checkpoints_wan2_p9_t49_c16_r7_100vid_stable/stage2_wan_lora_step_100_wan_trainable.pt
Free disk after save: 3.18 GB
[stage2_wan_lora] step 125/500 loss=1.7141 grad_norm=0.1221 lr=9.40e-07 skipped=0 peak_vram=26.69GB
[stage2

## Stage 5.1 — Benchmark Evaluation

Benchmark dùng reconstruction từ Wan2.2-backed latent model. LPIPS là LPIPS thật (`lpips_alex`). Notebook này không còn fallback sang L1 proxy. `worldscore_*_proxy` vẫn chỉ là proxy vì chưa có evaluator WorldScore chính thức.


In [33]:
import os, glob, shutil, subprocess, sys

# Tìm Kaggle Model lpips-offline
roots = glob.glob("/kaggle/input/models/**/lpips-offline/pytorch/default/*", recursive=True)
roots = [p for p in roots if os.path.isdir(p)]

print("LPIPS roots:")
for p in roots:
    print("-", p)

if not roots:
    raise FileNotFoundError("Chưa add Kaggle Model lpips-offline vào notebook.")

LPIPS_ROOT = roots[0]
WHEEL_DIR = os.path.join(LPIPS_ROOT, "wheels")
TORCH_CACHE_SRC = os.path.join(LPIPS_ROOT, "torch")
TORCH_CACHE_DST = "/root/.cache/torch"

print("LPIPS_ROOT:", LPIPS_ROOT)
print("WHEEL_DIR:", WHEEL_DIR)
print("TORCH_CACHE_SRC:", TORCH_CACHE_SRC)

if not os.path.isdir(WHEEL_DIR):
    raise FileNotFoundError(f"Không thấy wheels folder: {WHEEL_DIR}")

# Copy torch cache chứa pretrained alexnet/vgg/squeeze weights nếu có
if os.path.isdir(TORCH_CACHE_SRC):
    if os.path.exists(TORCH_CACHE_DST):
        shutil.rmtree(TORCH_CACHE_DST)
    shutil.copytree(TORCH_CACHE_SRC, TORCH_CACHE_DST)
    print("Copied torch cache to:", TORCH_CACHE_DST)
else:
    print("WARN: Không thấy torch cache trong LPIPS model input.")

print("\nAvailable wheels:")
!ls -lh "{WHEEL_DIR}"

# Cài LPIPS offline, KHÔNG resolve dependency torch/torchvision
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--no-index",
    "--find-links", WHEEL_DIR,
    "--no-deps",
    "--force-reinstall",
    "lpips"
])

print("Installed lpips with --no-deps")

LPIPS roots:
- /kaggle/input/models/nhtdngtrn/lpips-offline/pytorch/default/1
LPIPS_ROOT: /kaggle/input/models/nhtdngtrn/lpips-offline/pytorch/default/1
WHEEL_DIR: /kaggle/input/models/nhtdngtrn/lpips-offline/pytorch/default/1/wheels
TORCH_CACHE_SRC: /kaggle/input/models/nhtdngtrn/lpips-offline/pytorch/default/1/torch
Copied torch cache to: /root/.cache/torch

Available wheels:
total 56K
-rw-r--r-- 1 nobody nogroup 53K Jun 20 04:44 lpips-0.1.4-py3-none-any.whl
Looking in links: /kaggle/input/models/nhtdngtrn/lpips-offline/pytorch/default/1/wheels
Processing /kaggle/input/models/nhtdngtrn/lpips-offline/pytorch/default/1/wheels/lpips-0.1.4-py3-none-any.whl
Installed lpips with --no-deps


In [34]:
import torch
import lpips

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("torch:", torch.__version__)
print("torchvision import test:")
import torchvision
print("torchvision:", torchvision.__version__)

LPIPS_NET = "alex"
lpips_fn = lpips.LPIPS(net=LPIPS_NET).to(DEVICE).eval()
_HAS_LPIPS = True

@torch.no_grad()
def compute_lpips_video_01(pred01, target01, batch_frames=4):
    """pred01/target01: [T,C,H,W] or [B,T,C,H,W], range [0,1]."""
    if pred01.ndim == 5:
        pred01 = pred01.flatten(0, 1)
        target01 = target01.flatten(0, 1)
    pred = pred01.float().clamp(0, 1) * 2 - 1
    target = target01.float().clamp(0, 1) * 2 - 1
    vals = []
    for i in range(0, pred.shape[0], batch_frames):
        p = pred[i:i + batch_frames].to(DEVICE)
        t = target[i:i + batch_frames].to(DEVICE)
        vals.append(lpips_fn(p, t).view(-1).detach().cpu())
    return torch.cat(vals).mean().item()

x = torch.rand(1, 3, HEIGHT, WIDTH, device=DEVICE)
y = torch.rand(1, 3, HEIGHT, WIDTH, device=DEVICE)
print("LPIPS true test OK:", compute_lpips_video_01(x, y))


torch: 2.10.0+cu128
torchvision import test:
torchvision: 0.25.0+cu128
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.12/dist-packages/lpips/weights/v0.1/alex.pth
LPIPS true test OK: 0.19394272565841675


In [35]:
import types
import torch

def _tree_to_device_dtype(x, device, dtype):
    if torch.is_tensor(x):
        if x.is_floating_point():
            return x.to(device=device, dtype=dtype)
        return x.to(device=device)

    if isinstance(x, dict):
        return {k: _tree_to_device_dtype(v, device, dtype) for k, v in x.items()}

    if isinstance(x, list):
        return [_tree_to_device_dtype(v, device, dtype) for v in x]

    if isinstance(x, tuple):
        return tuple(_tree_to_device_dtype(v, device, dtype) for v in x)

    return x

def _extract_tensor_from_wan_output(out):
    if "extract_tensor_from_output" in globals():
        return extract_tensor_from_output(out)

    if torch.is_tensor(out):
        return out

    if hasattr(out, "sample") and torch.is_tensor(out.sample):
        return out.sample

    if isinstance(out, dict):
        for k in ["sample", "pred", "prediction", "hidden_states"]:
            if k in out and torch.is_tensor(out[k]):
                return out[k]

    if isinstance(out, (tuple, list)):
        for v in out:
            if torch.is_tensor(v):
                return v

    raise RuntimeError(f"Cannot extract tensor from transformer output: {type(out)}")

def _get_patch_embedding_dtype_device(tr):
    if not hasattr(tr, "patch_embedding"):
        raise RuntimeError("transformer.patch_embedding not found.")

    pe = tr.patch_embedding

    if hasattr(pe, "bias") and pe.bias is not None:
        dtype = pe.bias.dtype
        device = pe.bias.device
    elif hasattr(pe, "weight"):
        dtype = pe.weight.dtype
        device = pe.weight.device
    else:
        raise RuntimeError("Cannot detect patch_embedding dtype/device.")

    return dtype, device

def patch_wan_benchmark_dtype(model):
    if not hasattr(model, "transformer"):
        raise RuntimeError("model.transformer not found.")

    tr = model.transformer
    pe_dtype, pe_device = _get_patch_embedding_dtype_device(tr)

    print("Using patch_embedding dtype:", pe_dtype)
    print("Using patch_embedding device:", pe_device)

    # Extra guard: patch patch_embedding itself so even if something passes float32,
    # Conv3D will receive bf16/fp16 input.
    pe = tr.patch_embedding

    if not hasattr(pe, "_orig_forward_dtype_guard"):
        pe._orig_forward_dtype_guard = pe.forward

        def _patch_embedding_forward_guard(self, hidden_states):
            dtype, device = _get_patch_embedding_dtype_device(tr)
            hidden_states = hidden_states.to(device=device, dtype=dtype)
            return self._orig_forward_dtype_guard(hidden_states)

        pe.forward = types.MethodType(_patch_embedding_forward_guard, pe)
        print("Patched transformer.patch_embedding.forward dtype guard.")
    else:
        print("patch_embedding dtype guard already exists.")

    def _transformer_call_fixed(self, noisy_latents, t, prompt_embeds, grad_enabled=False):
        tr = self.transformer

        # Important: use patch_embedding dtype, not first transformer parameter dtype.
        tr_dtype, tr_device = _get_patch_embedding_dtype_device(tr)

        noisy_latents = noisy_latents.to(device=tr_device, dtype=tr_dtype)
        prompt_embeds = _tree_to_device_dtype(prompt_embeds, tr_device, tr_dtype)

        if torch.is_tensor(t):
            # timestep can stay float32, but must be on same device
            t = t.to(device=tr_device)

        kwargs = {
            "hidden_states": noisy_latents,
            "timestep": t,
            "encoder_hidden_states": prompt_embeds,
            "return_dict": False,
        }

        with torch.set_grad_enabled(grad_enabled):
            out = tr(**kwargs)

        pred = _extract_tensor_from_wan_output(out)

        if pred.shape != noisy_latents.shape:
            if pred.ndim == noisy_latents.ndim:
                slices = tuple(slice(0, min(a, b)) for a, b in zip(pred.shape, noisy_latents.shape))
                pred = pred[slices]

            if pred.shape != noisy_latents.shape:
                raise RuntimeError(
                    f"Transformer output shape mismatch: "
                    f"pred={tuple(pred.shape)}, expected={tuple(noisy_latents.shape)}"
                )

        return pred

    model._transformer_call = types.MethodType(_transformer_call_fixed, model)

    print("Patched model._transformer_call using patch_embedding dtype.")

patch_wan_benchmark_dtype(model)

Using patch_embedding dtype: torch.bfloat16
Using patch_embedding device: cuda:0
Patched transformer.patch_embedding.forward dtype guard.
Patched model._transformer_call using patch_embedding dtype.


In [36]:
import json, math
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

RUN_BENCHMARK = True
BENCHMARK_MAX_BATCHES = min(20, max(1, len(val_loader)))

try:
    from skimage.metrics import structural_similarity as _skimage_ssim
    _HAS_SKIMAGE = True
except Exception:
    _HAS_SKIMAGE = False

if "compute_lpips_video_01" not in globals():
    raise RuntimeError("True LPIPS function not found. Run the LPIPS setup cell first.")

def _to_01(x):
    return ((x.detach().float().clamp(-1, 1) + 1) / 2).clamp(0, 1)

def _psnr(pred, target):
    mse = F.mse_loss(pred, target).item()
    return -10 * math.log10(max(mse, 1e-8))

def _ssim_single(pred, target):
    if _HAS_SKIMAGE:
        p = pred.detach().cpu().permute(1, 2, 0).numpy()
        t = target.detach().cpu().permute(1, 2, 0).numpy()
        return float(_skimage_ssim(t, p, channel_axis=2, data_range=1.0))
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    mu_x = pred.mean(); mu_y = target.mean()
    var_x = pred.var(unbiased=False); var_y = target.var(unbiased=False)
    cov = ((pred - mu_x) * (target - mu_y)).mean()
    return float(((2 * mu_x * mu_y + C1) * (2 * cov + C2) / ((mu_x ** 2 + mu_y ** 2 + C1) * (var_x + var_y + C2))).detach().cpu())

def _ssim_video(pred, target, max_frames=8):
    T = min(pred.shape[0], max_frames)
    idxs = torch.linspace(0, pred.shape[0] - 1, steps=T).long().tolist()
    return float(np.mean([_ssim_single(pred[i], target[i]) for i in idxs]))

def _lpips_true(pred01, target01):
    return compute_lpips_video_01(pred01, target01)

@torch.no_grad()
def reconstruct_batch(model, batch, t_value=0.2):
    model.eval()
    return model.reconstruct_video(batch, t_value=t_value)

@torch.no_grad()
def run_benchmark(model, loader, max_batches=20):
    rows = []
    for bi, batch in enumerate(loader):
        if bi >= max_batches:
            break
        pred, target, batch = reconstruct_batch(model, batch, t_value=0.2)
        pred01 = _to_01(pred); target01 = _to_01(target)
        for b in range(pred01.shape[0]):
            pv = pred01[b]; tv = target01[b]
            row = {
                "id": batch["id"][b] if isinstance(batch.get("id"), list) else str(bi),
                "psnr": _psnr(pv, tv),
                "ssim": _ssim_video(pv, tv),
                "lpips": _lpips_true(pv, tv),
                "psnr_c": _psnr(pv[-1], tv[-1]),
                "ssim_c": _ssim_single(pv[-1], tv[-1]),
                "lpips_c": _lpips_true(pv[-1:], tv[-1:]),
            }
            rows.append(row)
    df = pd.DataFrame(rows)
    summary = {}
    if len(df):
        for col in ["psnr", "ssim", "lpips", "psnr_c", "ssim_c", "lpips_c"]:
            summary[col] = float(df[col].mean())
        static_score = float(np.clip(summary["ssim"] * 100, 0, 100))
        dynamic_score = float(np.clip((1 - summary["lpips"]) * 100, 0, 100))
        camera_ctrl = float(np.clip((summary["psnr_c"] / 30) * 100, 0, 100))
        avg_score = float(np.mean([static_score, dynamic_score, camera_ctrl]))
        summary.update({
            "worldscore_avg_proxy": avg_score,
            "worldscore_static_proxy": static_score,
            "worldscore_dynamic_proxy": dynamic_score,
            "worldscore_camera_ctrl_proxy": camera_ctrl,
            "lpips_source": "lpips_alex",
            "num_eval_samples": int(len(df)),
            "benchmark_note": "WorldScore values are proxy only; PSNR/SSIM/LPIPS are computed from reconstruction on validation samples.",
        })
    return df, summary

if RUN_BENCHMARK:
    benchmark_df, benchmark_summary = run_benchmark(model, val_loader, BENCHMARK_MAX_BATCHES)
    print("Benchmark summary:")
    print(json.dumps(benchmark_summary, indent=2))
    try:
        display(benchmark_df.head())
    except Exception:
        print(benchmark_df.head())
else:
    benchmark_df = pd.DataFrame()
    benchmark_summary = {"skipped": True}
    print("Benchmark skipped")


Benchmark summary:
{
  "psnr": 24.120324809373948,
  "ssim": 0.7604297330603004,
  "lpips": 0.15584860779345036,
  "psnr_c": 23.554185994533128,
  "ssim_c": 0.7493908494710922,
  "lpips_c": 0.18351222425699235,
  "worldscore_avg_proxy": 79.65735528059848,
  "worldscore_static_proxy": 76.04297330603004,
  "worldscore_dynamic_proxy": 84.41513922065496,
  "worldscore_camera_ctrl_proxy": 78.51395331511043,
  "lpips_source": "lpips_alex",
  "num_eval_samples": 20,
  "benchmark_note": "WorldScore values are proxy only; PSNR/SSIM/LPIPS are computed from reconstruction on validation samples."
}


,id,psnr,ssim,lpips,psnr_c,ssim_c,lpips_c
0,00117_0ce3839aa5b66e3f,25.626368,0.815210,0.150692,23.921330,0.779935,0.168524
1,00118_0cf444aef3ba16bd,25.197163,0.785901,0.141488,24.968846,0.778054,0.166014
2,00119_0d01d4d6c5d5297e,25.781610,0.816436,0.153646,24.812482,0.828413,0.200457
3,00120_0d06be83296cf911,21.798814,0.729386,0.138298,20.725183,0.672881,0.165445
4,00121_0d08611c8b251e15,23.052750,0.756819,0.171857,23.284091,0.758951,0.190631


## Stage 5.2 — Quick Proxy Inference Sample


In [37]:
def tensor_to_uint8_video(x):
    x = ((x.detach().cpu().clamp(-1, 1) + 1) * 127.5).byte().numpy()
    return np.transpose(x, (0, 2, 3, 1))

@torch.no_grad()
def save_proxy_sample(model, loader, out_path, t_value=0.2):
    model.eval()
    batch = next(iter(loader))
    pred, target, _ = model.reconstruct_video(batch, t_value=t_value)
    vid = tensor_to_uint8_video(pred[0])
    out_path = Path(out_path)
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, 8, (vid.shape[2], vid.shape[1]))
    for fr in vid:
        writer.write(cv2.cvtColor(fr, cv2.COLOR_RGB2BGR))
    writer.release()
    model.train()
    return out_path

sample_path = save_proxy_sample(model, val_loader, SAMPLE_DIR / "wan_spatia_reconstruction.mp4", t_value=0.2)
print("Saved sample:", sample_path)


Saved sample: /kaggle/working/spatia_full_samples_wan2_p9_t49_c16_r7_100vid_stable/wan_spatia_reconstruction.mp4


## Stage 6.1 — Final Save Checkpoint, Config, and Evaluation Metadata


In [38]:
final_ckpt = CKPT_DIR / "spatia_wan2_trainable_delta_final.pt"
config_path = CKPT_DIR / "spatia_wan2_config.json"

payload = {
    "trainable_state": model.export_trainable_state(),
    "prompt_to_id": train_ds.prompt_to_id,
    "config": {
        "height": HEIGHT,
        "width": WIDTH,
        "prev_frames": PREV_FRAMES,
        "target_frames": TARGET_FRAMES,
        "candidate_frames": CANDIDATE_FRAMES,
        "ref_frames": REF_FRAMES,
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "amp_dtype": str(AMP_DTYPE),
        "effective_batch": BATCH_SIZE * GRAD_ACCUM_STEPS,
        "lr_stage1": LR_STAGE1,
        "lr_stage2": LR_STAGE2,
        "grad_clip_stage1": GRAD_CLIP_STAGE1,
        "grad_clip_stage2": GRAD_CLIP_STAGE2,
        "control_width": CONTROL_WIDTH,
        "control_depth": CONTROL_DEPTH,
        "control_output_scale": CONTROL_OUTPUT_SCALE,
        "wan_dir": str(WAN_DIR),
        "default_prompt": DEFAULT_PROMPT,
        "note": "Wan2.2-backed Spatia-style training. Saves control branch + Wan LoRA delta only, not full Wan2.2 base weights.",
        "benchmark_summary": globals().get("benchmark_summary", {}),
    },
}
safe_torch_save(payload, final_ckpt, min_free_gb=MIN_FREE_GB_FOR_SAVE)

with open(config_path, "w", encoding="utf-8") as f:
    json.dump({
        "train_videos": len(train_ds),
        "test_videos": len(val_ds),
        "height": HEIGHT,
        "width": WIDTH,
        "prev_frames": PREV_FRAMES,
        "target_frames": TARGET_FRAMES,
        "candidate_frames": CANDIDATE_FRAMES,
        "ref_frames": REF_FRAMES,
        "stage1_steps": MAX_TRAIN_STEPS_STAGE1,
        "stage2_steps": MAX_TRAIN_STEPS_STAGE2,
        "paper_stage1_steps": PAPER_STAGE1_STEPS,
        "paper_stage2_steps": PAPER_STAGE2_STEPS,
        "lora_rank": LORA_RANK,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "amp_dtype": str(AMP_DTYPE),
        "effective_batch": BATCH_SIZE * GRAD_ACCUM_STEPS,
        "lr_stage1": LR_STAGE1,
        "lr_stage2": LR_STAGE2,
        "grad_clip_stage1": GRAD_CLIP_STAGE1,
        "grad_clip_stage2": GRAD_CLIP_STAGE2,
        "val_is_train_proxy": globals().get("VAL_IS_TRAIN_PROXY", None),
        "toggles": {
            "run_keye": RUN_KEYE,
            "run_referdino": RUN_REFERDINO,
            "run_mapanything": RUN_MAPANYTHING,
            "strict_external_models": STRICT_EXTERNAL_MODELS,
            "use_wan2_backbone": USE_WAN2_BACKBONE,
            "strict_wan_backbone": STRICT_WAN_BACKBONE,
        },
        "benchmark_summary": globals().get("benchmark_summary", {}),
        "pipeline_inputs": {
            "data_root": str(DATA_ROOT),
            "wan_model": str(WAN_MODEL),
            "wan_dir": str(WAN_DIR),
            "mapanything_model": str(MAPANYTHING_MODEL),
            "keye_model": str(KEYE_MODEL),
            "referdino_ckpt": str(REFERDINO_CKPT),
            "mapanything_repo": str(MAPANYTHING_REPO),
            "referdino_repo": str(REFERDINO_REPO),
        },
    }, f, indent=2)

print("Saved final trainable delta:", final_ckpt)
print("Saved config:", config_path)

if "benchmark_df" in globals() and len(benchmark_df):
    benchmark_csv = CKPT_DIR / "benchmark_results.csv"
    benchmark_json = CKPT_DIR / "benchmark_summary.json"
    benchmark_df.to_csv(benchmark_csv, index=False)
    with open(benchmark_json, "w", encoding="utf-8") as f:
        json.dump(benchmark_summary, f, indent=2)
    print("Saved benchmark CSV:", benchmark_csv)
    print("Saved benchmark summary:", benchmark_json)


Free disk before save: 3.18 GB
Saved /kaggle/working/spatia_full_checkpoints_wan2_p9_t49_c16_r7_100vid_stable/spatia_wan2_trainable_delta_final.pt
Free disk after save: 2.87 GB
Saved final trainable delta: /kaggle/working/spatia_full_checkpoints_wan2_p9_t49_c16_r7_100vid_stable/spatia_wan2_trainable_delta_final.pt
Saved config: /kaggle/working/spatia_full_checkpoints_wan2_p9_t49_c16_r7_100vid_stable/spatia_wan2_config.json
Saved benchmark CSV: /kaggle/working/spatia_full_checkpoints_wan2_p9_t49_c16_r7_100vid_stable/benchmark_results.csv
Saved benchmark summary: /kaggle/working/spatia_full_checkpoints_wan2_p9_t49_c16_r7_100vid_stable/benchmark_summary.json


## Stage 6.2 — Package Outputs


In [ ]:
zip_base = WORK_DIR / "spatia_100videos_rtx6000pro_stable_outputs"
if zip_base.with_suffix(".zip").exists():
    zip_base.with_suffix(".zip").unlink()

pack_dir = WORK_DIR / "spatia_full_package"
if pack_dir.exists():
    shutil.rmtree(pack_dir)
pack_dir.mkdir(parents=True, exist_ok=True)
shutil.copytree(CKPT_DIR, pack_dir / "checkpoints")
shutil.copytree(SAMPLE_DIR, pack_dir / "samples")

# Save a small manifest of processed sample metadata, not the heavy tensors.
meta_rows = []
for f in sorted(PROC_DIR.glob("*.pt")):
    item = torch.load(f, map_location="cpu")
    meta_rows.append({
        "id": item.get("id"),
        "prompt": item.get("prompt"),
        "entities": ", ".join(item.get("entities", [])),
        "video_path": item.get("video_path"),
        "pose_path": item.get("pose_path"),
        "keye_source": item.get("module_meta", {}).get("keye", {}).get("source"),
        "referdino_source": item.get("module_meta", {}).get("referdino", {}).get("source"),
        "mapanything_source": item.get("module_meta", {}).get("mapanything", {}).get("source"),
    })
pd.DataFrame(meta_rows).to_csv(pack_dir / "processed_manifest.csv", index=False)

shutil.make_archive(str(zip_base), "zip", root_dir=pack_dir)
print("Created:", zip_base.with_suffix(".zip"))
if zip_base.with_suffix(".zip").exists():
    print("Size MB:", zip_base.with_suffix(".zip").stat().st_size / 1024**2)

## Appendix — Mapping to Paper


## Stage 6.2 — Paper Compliance Audit

In [ ]:
print("===== PAPER COMPLIANCE AUDIT =====")
print("Training model class:", type(model).__name__)
print("Wan pipeline class:", type(model.pipe).__name__)
print("Transformer class:", type(model.transformer).__name__)
print("VAE class:", type(model.vae).__name__)
print("Text encoder class:", type(model.text_encoder).__name__)

wan_params = count_module_params(model.transformer) + count_module_params(model.vae) + count_module_params(model.text_encoder)
control_params = 0 if model.control is None else count_module_params(model.control)
lora_params = sum(p.numel() for n,p in model.transformer.named_parameters() if "lora" in n.lower())
print("Wan component params:", f"{wan_params/1e9:.3f}B")
print("Control params:", control_params)
print("LoRA params:", lora_params)
print("LoRA rank:", LORA_RANK)
print("AMP dtype:", AMP_DTYPE)
print("Stage LR:", LR_STAGE1, LR_STAGE2)
print("Effective batch:", BATCH_SIZE * GRAD_ACCUM_STEPS)
print("Val proxy:", globals().get("VAL_IS_TRAIN_PROXY", None))
print("Frames: prev/target/candidate/ref =", PREV_FRAMES, TARGET_FRAMES, CANDIDATE_FRAMES, REF_FRAMES)
print("Resolution:", HEIGHT, WIDTH)
print("Train/Test:", len(train_ds), len(val_ds))
print("LPIPS source:", globals().get("benchmark_summary", {}).get("lpips_source"))

if type(model).__name__ != "WanSpatiaTrainer":
    raise RuntimeError("Wrong model class. This is not the strict Wan2.2 notebook.")
if wan_params < 1_000_000_000:
    raise RuntimeError("Wan backbone parameter audit failed.")
if LORA_RANK != 64:
    raise RuntimeError("LoRA rank is not paper-aligned rank 64.")
if PREV_FRAMES != 9:
    raise RuntimeError("PREV_FRAMES is not 9.")
if REF_FRAMES != 7:
    raise RuntimeError("REF_FRAMES is not 7.")
print("Audit OK: notebook is using Wan2.2-backed training, not the old 0.888M toy adapter.")
